    # запрос по БАЛАНСОВЫМ транзакциям из MySQL
        # Создаём ДФ БАЛАНСОВЫХ операций

    # запрос по ТОРГОВЫМ транзакциям из MySQL
        # Создаём ДФ ТОРГОВЫХ операций

    # запрос ВСЕХ символов из MySQL
        # ФорМИрование таблицы спецификаций уникальных символов задействованных в торговле

    # Выявление id уникальных символов задействованных в торговле
        Таблица спецификаций ФОРМАТИРОВАННЫХ символов CRM занятых в торговле

    # ФормаТИрование таблицы спецификаций уникальных символов задействованных в торговле    
        # Убираем дублирующиеся символы, оставляя те, спецификации которых имеют более частое вхождение в торговую историю

    # Обьеденение Данных по торговым транзакциям и спецификаций инструментов 

    # Создаём Колонку с коэффициентом для пересчёта прибыли в балансовую валюту
    
    # формирование ОБЩЕЙ таблицы с торговыми и балансовыми результатами

    # Перенос Сделок.
        # Создание DF с итогами переноса позиций
        # Создание CSV с итогами переноса позиций
        
        
    

In [2]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("c:/unique_data/rep_fo_metatrader_server"))

# Сбрасываем кеш файлов с импортированными функциями <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
from imports import *
import imports, importlib
importlib.reload(imports)
print("Сбрасываем кеш файлов с импортированными функциями dir(imports) = ", dir(imports))

from formatting_transactions import *
import formatting_transactions, importlib
importlib.reload(formatting_transactions)
print("Сбрасываем кеш файлов с импортированными функциями dir(formatting_transactions) = ", dir(formatting_transactions))

from yar_sed_general_lib import *
import yar_sed_general_lib, importlib
importlib.reload(yar_sed_general_lib)
print("Сбрасываем кеш файлов с импортированными функциями dir(yar_sed_general_lib.py) = ", dir(yar_sed_general_lib))
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def id_list_from_file(file_path, delimiter=','):
    with open(file_path, 'r') as file:                                          # Читаем файл с идентификаторами
        id_list = file.read().strip().split(delimiter)                           # Извлекаем строки и делим их по запятой
    id_list = [int(id.strip()) for id in id_list]                       # Преобразуем идентификаторы в целые числа
    print(f"Длинна списка идентификаторов из файла {file_path}: {len(id_list)}")
    return id_list

Сбрасываем кеш файлов с импортированными функциями dir(imports) =  ['MT5Manager', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'admin_connect', 'admin_disconnect', 'array_element_attributes', 'attributes', 'attributes_last_tick', 'balance_0', 'connection', 'create_engine', 'create_engine_def', 'credits_dict_mt5manager', 'credits_dict_mysql', 'datetime', 'get_random_three_digits', 'getting_array_trading_instruments', 'importlib', 'imports_reset_cache', 'manager_connect', 'manager_disconnect', 'move_column', 'mt5admin', 'mt5manager', 'mysql', 'os', 'pd', 'pd_read_sql', 'pd_set_option', 'process_deal', 'random', 'sql_connection', 'sql_request', 'symbol_array_attributes', 'time', 'time_str_to_unix_time_2', 'timezone', 'update_column_based_on_list']
Сбрасываем кеш файлов с импортированными функциями dir(formatting_transactions) =  ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spe

In [3]:
# Функция для динамического импорта модулей <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
sys.path.append(os.path.abspath("c:/unique_data/rep_fo_metatrader_server"))
file_imports = "dynamic_import_functions.py"  # Файл с функциями для динамического импорта

if os.path.exists(file_imports):
    importlib.invalidate_caches()           # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info  # Импортируем только нужные функции
    print(f"Импорт [{file_imports}] успешен. \n Список импортированных функций:")
else: print(f"ERROR: Файл '{file_imports}' не найден, импорт не выполнен.")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

modules_to_import = {                                               # Словарь модулей для импорта
    "yar_sed_general_lib":
        ["c:/unique_data/rep_fo_metatrader_server",
                "list_print",
                "int_to_str_csv"]
                }

imported = import_functions(modules_to_import)                          # Импортируем модули из словаря modules_to_import
print_import_function_info(modules_to_import, imported)                 # Выводим переменные ожидаемые импортированными функциями 

Импорт [dynamic_import_functions.py] успешен. 
 Список импортированных функций:
Импорт из 'yar_sed_general_lib' успешен: ['list_print', 'int_to_str_csv']

 Импортированные функции и их параметры:
Функция 'list_print' из модуля 'yar_sed_general_lib' ожидает параметры: list, label=None
Функция 'int_to_str_csv' из модуля 'yar_sed_general_lib' ожидает параметры: list, short_file_path, file_name, time_in_name=True


In [3]:
# [ОБЯЗАТЕЛЕН] шаблоны запросов к  MySQL <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
query_balance = """SELECT DISTINCT d.account_id
FROM `br-stone`.`deposits` d
WHERE d.account_id IN (
    SELECT ca.account_id
    FROM `br-stone`.`customers_accounts` ca
    WHERE ca.account_id BETWEEN 7194 AND 99999999
    AND EXISTS (
        SELECT 1
        FROM `br-stone`.`deposits` d2
        WHERE d2.account_id = ca.account_id
        AND d2.deposit_date > '2023-06-01 00:00:00'
    )
)
AND d.deposit_status = 'approved' 
AND d.deposit_date > '2023-06-01 00:00:00'
AND d.account_id NOT IN (
    SELECT ca.account_id
    FROM `br-stone`.`customers_accounts` ca
    WHERE ca.account_lable = 'demo'
)"""

query_acc_balance_0 = """
AND d.account_id IN (
    SELECT account_id
    FROM `br-stone`.`customers_accounts`
    WHERE balance = 0)"""

# Исключение счетов определённых в диалоге с метизовым
error_acc_1 = [233209, 233231, 233237, 33993]                                       # Счета определённые как ошибочные в процессе исследования БД ЦРМ
error_acc_2 = [75203, 143301, 106952, 33993, 43532, 98124, 143886, 58448, 168912, 
               233240, 82138, 9948, 149667, 115190, 200762, 91902]                  # Исключение счетов у которых сделки по символам без конфигураций
error_acc_3 = [233209, 233233,  58448, 179936, 157572]                              # Прочие аккаунты с ошибками
# Счета по которым обнаружены сделки бес соответствующих символов на стороне MT5
error_acc_4 = [233209, 233231, 43532, 217701, 9948, 102457, 127656, 225344, 222868, 181953, 71582, 198274, 102606, 112145, 83421, 211194, 74174, 69200, 183419, 176370, 179097, 200034, 51617, 115190, 215234, 223494, 85046, 217416, 103422, 50754, 179936, 124529, 200762, 57435, 182557, 115904, 101385, 169448, 55596, 180969, 234213, 64492, 234754, 106952, 227970, 110405, 230039, 17782, 231878, 235002, 234798, 86354, 97843, 104956, 228334, 236282, 91902, 192527, 201601, 235267, 134593, 206724, 202530, 91552, 236294, 97726, 87405, 86067, 235819, 220338, 94739, 212436, 107303, 82923, 182835, 162580, 226468, 212547, 228261, 216191, 136921, 204892, 186863, 233254, 235265, 169436, 208085, 152524, 233188, 235748, 235755, 207544, 169893, 237627, 145027, 199523, 185101, 238075, 178869, 233993, 235006, 236868, 186111, 237119, 117167, 149875, 210959, 217943, 236970, 227148, 237131, 234970, 239407, 238644, 237471, 238625, 237080, 96306, 75140, 206039, 237663, 235525, 200647, 69806, 163161, 238062, 167288, 238476, 237658, 237670, 237787, 237335, 199268, 232713, 238467, 238384, 204521, 238702, 238034, 236014, 88241, 232388, 239701, 239253, 237875, 236292, 236499, 239025, 238577, 241765, 218624, 165232, 243419, 237132, 242399, 233233, 242446, 239101, 184600, 238729, 233237, 237395, 205027, 237129, 206499, 235351, 239430, 248754, 235671, 241901, 231493, 212945, 240948, 237691, 240962, 235296, 237569, 54989, 235860, 85806, 249187, 184419, 238292, 197717, 249420, 249159, 244630, 240779, 256283, 237118, 249058, 255871, 242545, 211211, 239045, 240838, 238217, 219203, 240986, 161625, 237710, 239277, 159445, 68295, 238655, 240609, 196743, 239875, 236017, 233110, 117622, 241787, 236932, 248048, 231117, 256008, 238800, 242915, 247900, 242355, 238262, 256369, 255170, 261054, 222590, 249131, 221874]
error_acc_5 = [93462, 117322, 192864, 234035, 84761]
error_acc_6 = [7194, 13665]
error_acc_7 = [70714, 53947]                                                        # https://fintech-area.slack.com/archives/G0165A4V6M7/p1737102160666239
error_acc_8 = [233272, 80620]                                                       # https://fintech-area.slack.com/archives/G0165A4V6M7/p1737827570914439
error_acc_9 = [30017, 233243, 237565, 233234, 233235, 233236, 233246, 233247]       # счета с валютой баланса НЕ USD

error_acc_set = set(error_acc_1 + error_acc_2 + error_acc_3 + error_acc_4 + error_acc_5 + error_acc_6 + error_acc_7 + error_acc_8 + error_acc_9)
print(f"[{len(error_acc_set)}] Счетов к ИСКЛЮЧЕНИЮ из миграции {error_acc_set}")

"""check_list_acc = [233272, 115190, 80620, 235705, 69806, 248824, 154719, 151997, 238639, 227970]

not_in_errors = [acc for acc in check_list_acc if acc not in error_acc_set]

if not_in_errors:
    print(f"Значения, которые НЕ входят в error_acc_set: {not_in_errors}")
else:
    print("Все значения из check_list_acc есть в error_acc_set")"""

#print(f"[{len(account_ids_crm)}] Счетов к МИГРАЦИИ {account_ids_crm}")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

[257] Счетов к ИСКЛЮЧЕНИЮ из миграции {218624, 237569, 235525, 256008, 101385, 233993, 43532, 143886, 192527, 210959, 112145, 94739, 237080, 7194, 238625, 96306, 97843, 86067, 182835, 85046, 238644, 234035, 102457, 200762, 237627, 70714, 237118, 237119, 225344, 238655, 50754, 212547, 219203, 231493, 237129, 117322, 237131, 237132, 249420, 58448, 69200, 239701, 197717, 237658, 57435, 204892, 247900, 237663, 184419, 199268, 217701, 237670, 241765, 238702, 124529, 183419, 237691, 241787, 216191, 198274, 227970, 145027, 196743, 238729, 238217, 240779, 237710, 222868, 239253, 233110, 230039, 235671, 149667, 226468, 206499, 235006, 127656, 239277, 69806, 88241, 220338, 242355, 221874, 178869, 238262, 207544, 53947, 115904, 181953, 215234, 255170, 240838, 68295, 33993, 54989, 102606, 231117, 238800, 238292, 208085, 159445, 206039, 136921, 82138, 237787, 9948, 243419, 242399, 179936, 249058, 205027, 233188, 234213, 235748, 242915, 180969, 204521, 235755, 80620, 241901, 248048, 237565, 176370, 

'check_list_acc = [233272, 115190, 80620, 235705, 69806, 248824, 154719, 151997, 238639, 227970]\n\nnot_in_errors = [acc for acc in check_list_acc if acc not in error_acc_set]\n\nif not_in_errors:\n    print(f"Значения, которые НЕ входят в error_acc_set: {not_in_errors}")\nelse:\n    print("Все значения из check_list_acc есть в error_acc_set")'

In [4]:
# [ОБЯЗАТЕЛЕН] Формирование списка счетов к миграции <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
#account_ids_crm = pd_read_sql(query_balance + query_acc_balance_0 + """;""")        # Запрашиваем  из CRM
account_ids_crm = pd_read_sql(query_balance  + """;""")                     # Запрашиваем  из CRM
account_ids = account_ids_crm['account_id'].tolist()                        # Формируем Список Счетов к миграции
account_ids = [x for x in account_ids if x not in error_acc_set]            # Удаляем счета помеченные как счета с ошибками
print(f"К миграции [{len(account_ids)}] торговых счетов: {account_ids}")

file_path = os.path.abspath('files/acc_list.csv')  
acc_set = list({int(x) for x in account_ids})                               # Преобразуем значения множества в целые числа и формируем список
open(file_path, 'w').close()                                                # Создаем пустой файл # Открываем файл в режиме записи и сразу закрываем
with open(file_path, 'w') as f:                                             # Открываем файл для записи
    f.write(', '.join(map(str, acc_set)))                                   # Преобразуем элементы множества в строки и записываем их, разделяя запятыми

file_path = os.path.abspath('files/acc_list.csv')                           # Получаем абсолютный путь к файлу
print(f"Полный путь к файлу со списком счетов к миграции [{file_path}]")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

К миграции [1566] торговых счетов: [163012, 223157, 227441, 230589, 139807, 37788, 90054, 127717, 215027, 217778, 229213, 196889, 50548, 216159, 231205, 163878, 136934, 216193, 57730, 151198, 64751, 218543, 195740, 77018, 59025, 115229, 63237, 127310, 169930, 60191, 135796, 151997, 114014, 69177, 77747, 187498, 211323, 229419, 51273, 221162, 150765, 187488, 140805, 140338, 180429, 61198, 150072, 81429, 93586, 110312, 150901, 226955, 73740, 132596, 104486, 73852, 137017, 138117, 215309, 116066, 70223, 135507, 71025, 212930, 209483, 193728, 86327, 69530, 101841, 130601, 59947, 55627, 211692, 95948, 87057, 60224, 67074, 107559, 120253, 143559, 76728, 84054, 210939, 41615, 75286, 71820, 93860, 132108, 78910, 178998, 71921, 165620, 140613, 67427, 91898, 70133, 232136, 65649, 111229, 68257, 87233, 232119, 86853, 217630, 143820, 231881, 222897, 82283, 217813, 156971, 56990, 74114, 71880, 70752, 204014, 206442, 191504, 55887, 192776, 80912, 219426, 177637, 209717, 132761, 115232, 161520, 83187

In [5]:
# Получение данных из таблицы `br-stone`.customers_accounts <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
all_acc_df = pd_read_sql("""SELECT account_id FROM `br-stone`.customers_accounts;""") # Cоздаём ДФ со всеми счетами
all_acc_list = all_acc_df['account_id'].dropna().astype(int).tolist()  # Убираем NaN и приводим к int # Преобразование столбца в список
del all_acc_df # Очистка памяти
all_acc_set = set(all_acc_list)             # Преобразование списка в множество уникальных значений
off_acc_set = all_acc_set - set(acc_set)    # Список счетов, которые не участвуют в миграции

print(f"[{len(all_acc_set)}] Счетов всего \n,"
      f"[{len(off_acc_set)}] Счетов к исключению из миграции \n,"
      f"[{len(acc_set)}] Счетов к миграции")

open('files/acc_list.csv', 'w').close()                                         # Создаем пустой файл # Открываем файл в режиме записи и сразу закрываем
with open('files/acc_list.csv', 'w') as f:                                      # Открываем файл для записи
    f.write(', '.join(map(str, acc_set)))                                       # Преобразуем элементы множества в строки и записываем их, разделяя запятыми
file_path = os.path.abspath('files/acc_list.csv')                               # Получаем абсолютный путь к файлу
print(f"Полный путь к файлу со списком счетов к миграции [{file_path}]")

[268478] Счетов всего 
,[266912] Счетов к исключению из миграции 
,[1566] Счетов к миграции
Полный путь к файлу со списком счетов к миграции [c:\unique_data\rep_fo_metatrader_server\files\acc_list.csv]


In [3]:
# Задаём ограгиченный список счетов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
acc_set = [265144]

In [6]:
# [ОБЯЗАТЕЛЕН] запрос по БАЛАНСОВЫМ и РЕФЕРАЛЬНЫМ транзакциям из MySQL <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
imported['int_to_str_csv'](acc_set,  "file\\temp",  "_acc_set.csv")

acc_list_str = ', '.join(map(str, sorted(acc_set)))                            # Преобразуем список счетов в строку и упорядываючем по возрастанию

# Запрос по БАЛАНСОВЫМ операциям из CRM <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
query_balance = f"""SELECT * FROM `br-stone`.`deposits` d WHERE d.account_id IN ({acc_list_str})
                    AND d.deposit_status = 'approved' 
                    AND d.deposit_date > '2023-05-31 23:59:59'
                    AND d.deposit_date < '2025-01-25 00:00:00';"""
balance_df = pd_read_sql(query_balance)                                                             # Выполнение SQL-запроса и создание DataFrame
balance_df = format_balance_trans(balance_df)                                                       # Форматирование ДФ балансовых операций

# Запрос по начислениям от РЕФЕРАЛОВ из MySQL; Создание ДФ; Форматирование ДФ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
referrals_withdrawals = f"""SELECT * FROM `br-stone`.`referrals_withdrawals` WHERE mt_id IN ({acc_list_str});"""
referrals_withdrawals_df = pd_read_sql(referrals_withdrawals)                                         # Выполнение SQL-запроса и создание DataFrame
referrals_withdrawals_df['Time'] = referrals_withdrawals_df['created_at'].astype('int64') // 10**9  # Преобразование в Unix time (секунды)
referrals_withdrawals_df['TimeMsc']  = referrals_withdrawals_df['Time']* 1000# + get_random_three_digits()
referrals_withdrawals_df = referrals_withdrawals_df.rename(columns={'withdrawal_amount_in_n_currency': 'balance_transactions'})
referrals_withdrawals_df = referrals_withdrawals_df.rename(columns={'mt_id': 'account_id'})
referrals_withdrawals_df['comment'] = "referrals"
referrals_withdrawals_df['finance_type'] = "correction"
referrals_withdrawals_df['type'] = "balance"
referrals_withdrawals_df.drop(columns=['updated_at'], inplace=True)
referrals_withdrawals_df.drop(columns=['lead_id'], inplace=True)
referrals_withdrawals_df.drop(columns=['created_at'], inplace=True)

# Объединение БАЛАНСОВЫХ и РЕФЕРАЛЬНЫХ начислений <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

# Функция для обработки списка DataFrame: удаляет полностью пустые строки и столбцы <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
def clean_and_concat(dataframes):
    """Функция для обработки списка DataFrame: удаляет полностью пустые строки и столбцы, а затем объединяет их в один DataFrame.
        Параметры:
            dataframes (list of pd.DataFrame): Список DataFrame для обработки и объединения.
        Возвращает:
            pd.DataFrame: Объединённый DataFrame с удалёнными пустыми строками и столбцами."""
   
    cleaned_dataframes = []
    for df in dataframes:                                                       # Обрабатываем каждый DataFrame в списке
        if not df.empty:                                                        # Пропускаем полностью пустые DataFrame
            df = df.dropna(how="all")                                           # Удаляем полностью пустые строки
            df = df.dropna(axis=1, how="all")                                   # Удаляем полностью пустые столбцы
            cleaned_dataframes.append(df)

    if cleaned_dataframes:                                                      # Проверяем, есть ли непустые DataFrame для объединения
        result_df = pd.concat(cleaned_dataframes, ignore_index=True)            # Объединяем обработанные DataFrame
    else: result_df = pd.DataFrame()                                            # Если все DataFrame были пустыми, возвращаем пустой DataFrame

    return result_df
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

#balance_df = pd.concat([balance_df, referrals_withdrawals_df], ignore_index=True)
balance_df = clean_and_concat([balance_df, referrals_withdrawals_df])           # Объединяем БАЛАНСОВЫЕ и РЕФЕРАЛЬНЫЕ начисления
balance_df.sort_values(by='TimeMsc', inplace=True)
unique_acc_balance = set(balance_df['account_id'].dropna())                     # Список счетов к миграции
text = f"\n Таблица Отформатированных БАЛАНСОВЫХ и РЕФЕРАЛЬНЫХ операций; торговых счетов: [{len(unique_acc_balance)}]"
pd_set_option(text, balance_df, 6)

Список из [1566] элементов, сохранён в файл: file\temp\2025-01-27 22-26-24.953_acc_set.csv

 Таблица Отформатированных БАЛАНСОВЫХ и РЕФЕРАЛЬНЫХ операций; торговых счетов: [1559]


,id,account_id,balance_transactions,comment,TimeMsc,Time,finance_type,type
0,42598,163012,1000000.00,1688373677,1688384477000,1688384477,correction,balance
1,42605,163012,1026473.00,1688553252,1688564052000,1688564052,correction,balance
2,42606,223157,500.00,1688559273,1688570073000,1688570073,deposit,balance
...,...,...,...,...,...,...,...,...
8435,231,243181,175.38,referrals,1737770738000,1737770738,correction,balance
8436,232,237728,180.87,referrals,1737810728000,1737810728,correction,balance
8437,233,239742,567.00,referrals,1738014455000,1738014455,correction,balance


In [40]:
#balance_df.to_csv("balance_d_202501180001.csv", index=False, encoding='utf-8')     


In [ ]:
balance_df = pd.read_csv("balance_d_202501180001.csv", encoding='utf-8')                 # Обратно загружаем данные из CSV
# Загружаем список из файла <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
def load_account_list(file_path):
    with open(file_path, 'r') as file:                                          # Читаем файл с идентификаторами
        account_ids = file.read().strip().split(', ')                           # Извлекаем строки и делим их по запятой
    account_ids = [int(id.strip()) for id in account_ids]                       # Преобразуем идентификаторы в целые числа
    print(f"Файл: {file_path}; Длинна списка: {len(account_ids)}")
    return  account_ids
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
file_path = r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\dif_balans_id_list.csv"
dif_balans_id_list = load_account_list(file_path)

file_path = r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\dif_referrals_id_list.csv"
dif_referrals_id_list = load_account_list(file_path)

# Исключение Уже существующих Балансовых операций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# Выбор строк по сложным условиям
filtered_df = balance_df[
    ((balance_df['id'].isin(dif_balans_id_list)) & (balance_df['comment'] != 'referrals')) |
    ((balance_df['id'].isin(dif_referrals_id_list)) & (balance_df['comment'] == 'referrals'))]


balance_df =  filtered_df.copy()
text = f"\n Таблица ЗАГРУЖЕННЫХ БАЛАНСОВЫХ и РЕФЕРАЛЬНЫХ операций;"
pd_set_option(text, balance_df, 6)

In [7]:
# [ОБЯЗАТЕЛЕН] запрос по ТОРГОВЫМ транзакциям из MySQL <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````

query_trade = f"""SELECT * FROM `br-stone`.`trades` t WHERE t.account_id IN ({acc_list_str})
                    AND t.position_id != 2
                    AND t. open_time > '2023-05-31 23:59:59'
                    AND t. open_time < '2025-01-25 00:00:00';"""
# Добавляем исключение по списку id торговых операций


# Создаём ДФ ТОРГОВЫХ операций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
statement_df = pd_read_sql(query_trade)                                     # Выполнение SQL-запроса и создание DataFrame
unique_currency_id_statement = set(statement_df['currency_id'].dropna())    # Уникальных Идентификаторов [currency_id] торговых инструментов
unique_symbol_statement = set(statement_df['symbol'].dropna())              # Уникальных ИМЁН [symbol] торговых инструментов
text = f"\n [statement_df] ДФ ТОРГОВЫХ операций, ПЕРВЫЙ запрос по списку счетов [acc_list_str] \n Уникальных Идентификаторов [currency_id] торговых инструментов unique_currency_id_statement [{len(unique_currency_id_statement)}] \n Уникальных ИМЁН [symbol] торговых инструментов unique_symbol_statement [{len(unique_symbol_statement)}]"
pd_set_option(text, statement_df, 3)

statement_df = format_trade_trans(statement_df)  # Форматирование ДФ ТОРГОВЫХ операций
pd_set_option("[statement_df] ДФ ТОРГОВЫХ операций, После первичного форматироваеия", statement_df, 3)

# Удаление не нужных сделок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
ord_list = [732, 1263, 1812, 2101, 2103, 3261]
for i in ord_list: statement_df = delete_rows_by_condition(statement_df, "order_id", i)             # Удаляем Не нужные сделки
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Форматирование ПЕРЕОТКРЫТЫХ сделок; Удаляем значения цены и времени закрытия в ПЕРЕОТКРЫТЫХ сделках; создание идентификатора переоткрытия <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
statement_df.insert(7, "re_open", int)                                                              # Создаём колонку с идентификатором  переоткрытия
statement_df["re_open"] = np.where((statement_df["position_id"] == 1) & ((statement_df["close_price"] != 0) | ~(statement_df["close_price"].isna())) & (~statement_df["close_time"].isna()), 1, 0)

# Удаляем значения цены и времени закрытия в ПЕРЕОТКРЫТЫХ сделках <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print(f"\n Удаляем значения цены и времени закрытия в [{(statement_df["re_open"] == 1).sum()}] ПЕРЕОТКРЫТЫХ сделках")
statement_df["close_price"] = np.where(statement_df["re_open"] == 1, 0, statement_df["close_price"])# Удаляем цену закрытия в переоткрытых позициях
statement_df.loc[statement_df["re_open"] == 1, "close_time"] = pd.NaT
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

#statement_df.rename(columns={'swap': 'deals_swap'}, inplace=True)                                   # создаём колонку для SWOP по сделке

"""# анализ типов данных в колонке ['open_time']
data_types_count = statement_df['open_time'].map(type).value_counts().to_dict()
print(f"Словарь типов и количества элементов по колонке ['open_time'] [{data_types_count}]")
string_rows = statement_df[statement_df['open_time'].map(type) == str]
display(string_rows)

statement_df = statement_df.sort_values(by="re_open", ascending=False, ignore_index=True)               # Сортировка
"""

#v Проверяем что бы к миграции были только закрытые сделки [position_id = 3] и открытые сделки p[osition_id = 1], должны отсутствовать отложенные ордера [position_id = 2]                             
print("\n Проверка типов сделок к миграции.\n Количество вхождений каждого уникального значения в столбце",statement_df["position_id"].value_counts(),"\n")

unique_acc_trade = set(statement_df['account_id'].dropna())                                         # Уникальных счетов в торговых операциях
text = f"\n [unique_acc_trade] Таблица Отформатированных ТОРГОВЫХ операций, торговых счетов: [{len(unique_acc_trade)}]"
pd_set_option(text, statement_df, 3)


 [statement_df] ДФ ТОРГОВЫХ операций, ПЕРВЫЙ запрос по списку счетов [acc_list_str] 
 Уникальных Идентификаторов [currency_id] торговых инструментов unique_currency_id_statement [321] 
 Уникальных ИМЁН [symbol] торговых инструментов unique_symbol_statement [321]


,order_id,ip,lead_id,account_id,currency_id,symbol,position_id,volume_lots,leverage,margin,spread,command,open_price,current_rate,close_price,open_time,close_time,close_at_profit,close_at_profit_type,close_at_loss,close_at_loss_type,start_at_price,profit,swap,sync_date,execute_in_future,execute_in_future_params,execute_in_future_processed
0,260,95.217.166.120,163014,163012,151,_BAYER,3,1.00,10,5.60,-0.01,BUY,51.405,50.685,50.685,2023-07-03 12:07:42,2023-07-04 10:00:21,NaN,2,NaN,2,NaN,-0.79,-2.83,2023-07-04 10:00:20,None,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133224,209249,2a02:6680:110d:39af:afb4:ba49:7ed6:d29f,267250,265987,101,PLATINUM,1,0.01,100,9.73,-2.80,BUY,973.000,963.100,NaN,2025-01-24 23:56:29,NaT,NaN,2,NaN,2,NaN,-12.70,NaN,2025-01-27 22:28:26,None,None,0


Количество УДАЛЕНИЕ ОТКЛЮЧЕНО символа '/': 73071
[statement_df] ДФ ТОРГОВЫХ операций, После первичного форматироваеия


,order_id,ip,lead_id,account_id,currency_id,symbol,position_id,volume_lots,leverage,margin,spread,command,open_price,current_rate,close_price,open_time,close_time,close_at_profit,close_at_profit_type,close_at_loss,close_at_loss_type,start_at_price,profit,swap,sync_date,execute_in_future,execute_in_future_params,execute_in_future_processed
0,260,95.217.166.120,163014,163012,151,_BAYER,3,1.00,10,5.60,-0.01,0,51.405,50.685,50.685,2023-07-03 12:07:42,2023-07-04 10:00:21,NaN,2,NaN,2,NaN,-0.79,-2.83,2023-07-04 10:00:20,None,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133224,209249,2a02:6680:110d:39af:afb4:ba49:7ed6:d29f,267250,265987,101,PLATINUM,1,0.01,100,9.73,-2.80,0,973.000,963.100,NaN,2025-01-24 23:56:29,NaT,NaN,2,NaN,2,NaN,-12.70,NaN,2025-01-27 22:28:26,None,None,0


Удалённо [0] строк в которых [order_id = 732]
Удалённо [0] строк в которых [order_id = 1263]
Удалённо [0] строк в которых [order_id = 1812]
Удалённо [1] строк в которых [order_id = 2101]
Удалённо [1] строк в которых [order_id = 2103]
Удалённо [0] строк в которых [order_id = 3261]

 Удаляем значения цены и времени закрытия в [98] ПЕРЕОТКРЫТЫХ сделках

 Проверка типов сделок к миграции.
 Количество вхождений каждого уникального значения в столбце position_id
3    126862
1      6361
Name: count, dtype: int64 


 [unique_acc_trade] Таблица Отформатированных ТОРГОВЫХ операций, торговых счетов: [1498]


,order_id,ip,lead_id,account_id,currency_id,symbol,position_id,re_open,volume_lots,leverage,margin,spread,command,open_price,current_rate,close_price,open_time,close_time,close_at_profit,close_at_profit_type,close_at_loss,close_at_loss_type,start_at_price,profit,swap,sync_date,execute_in_future,execute_in_future_params,execute_in_future_processed
0,260,95.217.166.120,163014,163012,151,_BAYER,3,0,1.00,10,5.60,-0.01,0,51.405,50.685,50.685,2023-07-03 12:07:42,2023-07-04 10:00:21,NaN,2,NaN,2,NaN,-0.79,-2.83,2023-07-04 10:00:20,None,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133224,209249,2a02:6680:110d:39af:afb4:ba49:7ed6:d29f,267250,265987,101,PLATINUM,1,0,0.01,100,9.73,-2.80,0,973.000,963.100,NaN,2025-01-24 23:56:29,NaT,NaN,2,NaN,2,NaN,-12.70,NaN,2025-01-27 22:28:26,None,None,0


In [ ]:
statement_df = pd.read_csv("statement_df_202501180001.csv", encoding='utf-8')                 # Обратно загружаем данные из CSV
# Загружаем список из файла <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
def load_account_list(file_path):
    with open(file_path, 'r') as file:                                          # Читаем файл с идентификаторами
        account_ids = file.read().strip().split(', ')                           # Извлекаем строки и делим их по запятой
    account_ids = [int(id.strip()) for id in account_ids]                       # Преобразуем идентификаторы в целые числа
    print(f"Файл: {file_path}; Длинна списка: {len(account_ids)}")
    return  account_ids
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
file_path = r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\dif_trade_id_list.csv"
dif_trade_id_list = load_account_list(file_path)

filtered_d = statement_df.copy()
# Выбор строк 
filtered_df = statement_df[statement_df['order_id'].isin(dif_trade_id_list)]


statement_df = filtered_df.copy()
text = f"\n Таблица ТОРГОВЫХ операций;"
pd_set_option(text, statement_df, 3)

In [9]:
statement_df.to_csv("statement_df_202501260002.csv", index=False, encoding='utf-8')      # Сохраняем данные сделок для переноса в CSV

In [ ]:
# исследуем базу на предмет изменённого плеча в открытой сделке <<<<<<<<<<<<<<<<<<<<<<<<<<<<
# Шаг 0: Фильтруем DataFrame по правилу position_id == 1
filtered_statement_df = statement_df[statement_df['position_id'] == 1]

# Шаг 1: Группируем по symbol и считаем количество уникальных значений leverage
symbols_with_multiple_leverages = (
    filtered_statement_df
    .groupby('symbol')['leverage']
    .nunique()
    .loc[lambda x: x > 1]  # Оставляем только те symbol, где leverage > 1
    .index  # Получаем список таких symbol
)

# Шаг 2: Фильтруем DataFrame, оставляя только строки с указанными символами
filtered_result_df = filtered_statement_df[
    filtered_statement_df['symbol'].isin(symbols_with_multiple_leverages)
]

# Шаг 3: Группируем и считаем количество leverage, создаём множество account_id
result_df = (
    filtered_result_df
    .groupby(['currency_id', 'symbol', 'leverage'])
    .agg(
        leverage_count=('leverage', 'size'),  # Количество leverage
        account_id_set=('account_id', lambda x: set(x))  # Множество значений account_id
    )
    .reset_index()  # Преобразуем в DataFrame
)

# Шаг 4: Добавляем колонку с количеством элементов в множестве
result_df['account_id_count'] = result_df['account_id_set'].apply(len)

pd_set_option("Проверка на разные плечи по одному и тому же инструменту", result_df, 50)

df_to_csv(result_df, "Проверка на разные плечи по одному и тому же инструменту.csv")


In [ ]:
# ЗАПРОС ПОКОНКРЕТНОМУ ОРДЕРУ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
filtr_df = statement_df[statement_df["order_id"] == 115419]
pd_set_option("ЗАПРОС ПОКОНКРЕТНОМУ ОРДЕРУ", filtr_df, 3)

In [8]:
# [ОБЯЗАТЕЛЕН] запрос ВСЕХ символов из MySQL <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
#query_symbols = """SELECT * FROM `br-stone`.`__currency` WHERE currency_id IN (SELECT currency_id FROM `br-stone`.`trades`);"""
unique_currency_id_statement_str = ', '.join(f"'{symbol}'" for symbol in unique_currency_id_statement)
query_symbols = f"""SELECT * FROM `br-stone`.`__currency` WHERE currency_id IN ({unique_currency_id_statement_str});"""

# Cоздание и форматирование таблицы символов CRM занятых в торговле <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
currency_crm_df = pd_read_sql(query_symbols)                                                    # Выполнение SQL-запроса и создание DataFrame
print("Первый запрос всей базы торговых инструментов; \n",
      f"уникальных значений [currency_crm_df['_symbol']]: [{len(set(currency_crm_df['_symbol'].dropna()))}]; \n",
      f"уникальных значений [currency_crm_df['currency_id']]: [{len(set(currency_crm_df['currency_id'].dropna()))}] \n",
      f"уникальных значений [currency_crm_df['currency_name']]: [{len(set(currency_crm_df['currency_name'].dropna()))}]")

currency_id_tuple = crm_symbol_tuple(statement_df)                                              # Выявление id уникальных символов задействованных в торговле
"""print(f" \n Уникальных ID символов задействованных в торговле[statement_df[currency_id]]: [{len(currency_id_tuple)}]; \n",
      f" \n Уникальных ID символов задействованных в торговле[statement_df]: [{len(currency_id_tuple)}]; \n")"""

symbols_to_create_df = currency_crm_df[currency_crm_df['currency_id'].isin(currency_id_tuple)]  # Отфильтровываем символы по currency_id
print(f" \n Количество уникальных значений [symbols_to_create_df['_symbol']] После удаления не занятых в торговле: [{len(set(symbols_to_create_df['_symbol'].dropna()))}].")

# Группируем по колонке '_symbol' и подсчитываем количество вхождений каждого значения
non_unique_symbols = symbols_to_create_df['_symbol'].value_counts()
# Выбираем только те значения, которые встречаются более одного раза
non_unique_symbols = non_unique_symbols[non_unique_symbols > 1].index
# Отбираем строки, где '_symbol' соответствует найденным неуникальным значениям
non_unique_df = symbols_to_create_df[symbols_to_create_df['_symbol'].isin(non_unique_symbols)]
pd_set_option("non_unique_symbols", non_unique_df, 10)

crm_symbol_df = symbols_to_create_df
"""symbols_to_create_df.loc[:, 'currency_name'] = symbols_to_create_df['currency_name'].str.replace('/', '')       # Удаление символа '/' в колонке 'currency_name' с использованием .loc[] для избежания предупреждения

# ФормаТИрование таблицы спецификаций уникальных символов задействованных в торговле <<<<<<<<<<<<<<<<<
search_cyrillic_characters(symbols_to_create_df)        # Проверка на наличие специфических символов
search_prohibited_characters(symbols_to_create_df)      # Поиск запрещённых спецсимволов в строках ДФ
replacing_prohibited_characters(symbols_to_create_df)   # Замена запрещённых символов на "_"""

pd_set_option("\n Таблица спецификаций ФОРМАТИРОВАННЫХ символов CRM занятых в торговле [symbols_to_create_df]", symbols_to_create_df, 3)
"""# Убираем дублирующиеся символы, оставляя те, спецификации которых имеют более частое вхождение в торговую историю <<<<<<<<<<<<<<<<<
unique_sorted_df = currency_counts_def(statement_df, symbols_to_create_df)                      #Создаём DF с символами CRM задействованными в торговле 
unique_sorted_df = unique_sorted_df.sort_values(by="symbol_contract_size", ascending=False)     # ascending=False для сортировки по убыванию
pd_set_option("\n [unique_sorted_df] Убираем дублирующиеся символы, оставляя те, спецификации которых имеют более частое вхождение в торговую историю", unique_sorted_df, 50)
crm_symbol_df = unique_sorted_df.copy(deep=True)  """

"""# ЗАПРОС ПОКОНКРЕТНОМУ ОРДЕРУ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
filtr_df = crm_symbol_df[crm_symbol_df["currency_name"] == "USD/AUD"]
pd_set_option("crm_symbol_df", filtr_df, 3)

# ЗАПРОС ПОКОНКРЕТНОМУ ОРДЕРУ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
filtr_df = statement_df[statement_df["symbol"] == "USD/AUD"]
pd_set_option("statement_df", filtr_df, 3)"""

Первый запрос всей базы торговых инструментов; 
 уникальных значений [currency_crm_df['_symbol']]: [316]; 
 уникальных значений [currency_crm_df['currency_id']]: [321] 
 уникальных значений [currency_crm_df['currency_name']]: [321]
Кортеж id торговых инструментов:  (1, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 15, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 35, 36, 39, 41, 42, 43, 45, 46, 48, 49, 51, 52, 54, 56, 60, 62, 63, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 96, 101, 102, 104, 105, 106, 108, 120, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 154, 155, 156, 158, 159, 160, 161, 162, 163, 164, 166, 169, 170, 176, 180, 287, 288, 290, 291, 295, 296, 297, 305, 308, 310, 311, 314, 315, 316, 329, 335, 338, 342, 344, 345, 347, 348, 353, 361, 362, 363, 365, 366, 372, 373, 374, 377, 378, 381, 397, 408, 

,currency_id,trading_group_id,parent_currency_id,currency_name,currency_key_lr,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_point_tech,symbol_multiply,symbol_calc_mode,symbol_contract_size,symbol_margin_initial,0_Open,0_Close,1_Open,1_Close,2_Open,2_Close,3_Open,3_Close,4_Open,4_Close,5_Open,5_Close,6_Open,6_Close,leverage,spread,swap_buy,swap_sell,minimum_change,commission,expiration,sell_last_value,buy_last_value,asset_id,asset_priority,asset_active,updated_at
10,13,1,13,ETH,ETH,ETHUSD,Ethereum vs. USD,USD,USD,2,0.01,1.00,100.0,2,10.0,0.5,0,1440,0,1440,0,1440,0,1440,0,1440,0,1440,0,1440,400,90,0,0,None,None,None,3909.9397,3909.9397,5,0,0,2024-12-13 12:00:20
65,86,1,86,#JP_MORGAN,#JP_MORGAN,JPM,,USD,USD,2,0.01,0.01,100.0,2,100.0,0.1,0,0,870,1260,870,1260,870,1260,870,1260,870,1260,0,0,10,45,0,0,None,None,None,264.6000,264.8300,3,0,1,2025-01-27 22:28:19
87,127,1,127,#BOA,#BOA,BAC,,USD,USD,2,0.01,0.01,100.0,2,100.0,0.1,0,0,870,1260,870,1260,870,1260,870,1260,870,1260,0,0,10,45,0,0,None,None,None,32.1700,32.2400,3,0,0,2024-11-11 15:46:09
91,131,1,131,#NETFLIX,#NETFLIX,NFLX,,USD,USD,2,0.01,0.01,100.0,2,100.0,0.1,0,0,870,1260,870,1260,870,1260,870,1260,870,1260,0,0,10,45,0,0,None,None,None,964.4100,965.2200,3,0,1,2025-01-27 22:28:19
108,148,1,148,_DEUTSCHE_B,_DEUTSCHE_B,DB,,EUR,EUR,2,0.01,0.01,100.0,2,100.0,0.1,0,0,420,990,420,990,420,990,420,990,420,990,0,0,10,45,0,0,None,None,None,18.9500,18.9680,6,0,1,2025-01-27 18:29:20
139,315,1,315,ETH/USD,ETHUSD,ETHUSD,Ethereum vs. USD,USD,USD,2,0.01,1.00,100.0,2,10.0,0.5,0,1440,0,1440,0,1440,0,1440,0,1440,0,1440,0,1440,400,90,0,0,None,None,None,3098.3400,3098.3500,5,0,1,2025-01-27 22:28:20
169,434,1,434,Netflix Inc,NFLX,NFLX,,USD,USD,2,0.01,0.01,100.0,2,100.0,0.1,0,0,870,1260,870,1260,870,1260,870,1260,870,1260,0,0,10,45,0,0,None,None,None,965.0700,965.0700,3,0,1,2025-01-27 22:28:19
181,447,1,447,JPMorgan Chase & Co,JPM,JPM,,USD,USD,2,0.01,0.01,100.0,2,100.0,0.1,0,0,870,1260,870,1260,870,1260,870,1260,870,1260,0,0,10,45,0,0,None,None,None,264.7600,264.7600,3,0,1,2025-01-27 22:26:59
182,448,1,448,Bank of America Corp,BAC,BAC,,USD,USD,2,0.01,0.01,100.0,2,100.0,0.1,0,0,870,1260,870,1260,870,1260,870,1260,870,1260,0,0,10,45,0,0,None,None,None,46.8600,46.8600,3,0,1,2025-01-27 22:27:39
187,453,1,453,Deutsche Bank AG,DB,DB,,USD,USD,2,0.01,0.01,100.0,2,100.0,0.1,0,0,870,1260,870,1260,870,1260,870,1260,870,1260,0,0,10,45,0,0,None,None,None,19.9900,19.9900,3,0,1,2025-01-27 22:28:19



 Таблица спецификаций ФОРМАТИРОВАННЫХ символов CRM занятых в торговле [symbols_to_create_df]


,currency_id,trading_group_id,parent_currency_id,currency_name,currency_key_lr,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_point_tech,symbol_multiply,symbol_calc_mode,symbol_contract_size,symbol_margin_initial,0_Open,0_Close,1_Open,1_Close,2_Open,2_Close,3_Open,3_Close,4_Open,4_Close,5_Open,5_Close,6_Open,6_Close,leverage,spread,swap_buy,swap_sell,minimum_change,commission,expiration,sell_last_value,buy_last_value,asset_id,asset_priority,asset_active,updated_at
0,1,1,1,EUR/USD,EURUSD,EURUSD,Euro vs US Dollar,USD,EUR,5,0.01,0.0001,100000.0,0,100000.0,0,0,0,0,1440,0,1440,0,1440,0,1440,0,1440,0,0,400,9,0,0,None,None,None,1.04911,1.04917,1,0,1,2025-01-27 22:28:19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,974,1,974,iShares TIPS Bond ETF,TIP,TIP,,USD,USD,3,0.01,0.0100,100.0,2,10.0,0.1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,10,1,1,0,None,None,None,107.92000,107.92000,3,0,1,2025-01-27 22:25:39


'# ЗАПРОС ПОКОНКРЕТНОМУ ОРДЕРУ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<\nfiltr_df = crm_symbol_df[crm_symbol_df["currency_name"] == "USD/AUD"]\npd_set_option("crm_symbol_df", filtr_df, 3)\n\n# ЗАПРОС ПОКОНКРЕТНОМУ ОРДЕРУ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<\nfiltr_df = statement_df[statement_df["symbol"] == "USD/AUD"]\npd_set_option("statement_df", filtr_df, 3)'

In [ ]:
# проверяем множество символов в торговых инструментах и в Таблице торговых транзакций
unique_symbol_trade      = set(statement_df['symbol'].dropna())                                     # множество имён торговых инструментов в торговых операциях
unique_symbol_symbol_currency_name = set(crm_symbol_df['currency_name'].dropna())                # множество имён торговых инструментов в таблице торговых инструментов ['currency_name']
print("\n проверяем множество символов в торговых инструментах и в Таблице торговых транзакций")
comparison_sets(unique_symbol_symbol_currency_name, unique_symbol_trade)

Если множества идентичны то для дальнейшей работы с символами используем [crm_symbol_df]

In [9]:
# [ОБЯЗАТЕЛЕН] Создаём ДФ таблицы с символами соответствия <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
file_path='files/Original/symbol_mapinr_.csv'                                               # CSV Файл с соответствиями символов CRM и MT5 
csv_loader = CSVLoader(file_path='files/Original/symbol_mapinr_.csv',                       # вызываем класс  Создаём ДФ из CSV Файла
                       delimiter=',', encoding='ISO-8859-1', df_name='symbol_mapinr_df')
symbol_mapinr_df = csv_loader.load_data()                                                   
pd_set_option("\n ДФ соответствия символов CRM & MT5 [symbol_mapinr_df]:",symbol_mapinr_df, 3)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

[class CSVLoader]: DataFrame 'symbol_mapinr_df' успешно создан из 'files/Original/symbol_mapinr_.csv'.

 ДФ соответствия символов CRM & MT5 [symbol_mapinr_df]:


,maping_currency_name,maping_currency_key_lr,maping_symbol,maping_DB_SYMBOL
0,GBP/CHF,GBP/CHF,GBP/CHF,GBPCHF
...,...,...,...,...
291,Gazprom PJSC,GAZP,GAZP,OGZPY.MCS


In [ ]:
def find_max_intersection(list1, list2):
    result = []
    for i, set1 in enumerate(list1):
        max_intersection_count = 0
        best_match = None
        for j, set2 in enumerate(list2):
            intersection_count = len(set1 & set2)
            if intersection_count > max_intersection_count:
                max_intersection_count = intersection_count
                best_match = j
        result.append((i, best_match, max_intersection_count))
    return result

unique_maping_maping_currency_name      = set(symbol_mapinr_df['maping_currency_name'].dropna())
unique_maping_maping_currency_key_lr    = set(symbol_mapinr_df['maping_currency_key_lr'].dropna())
unique_maping_maping_symbol	            = set(symbol_mapinr_df['maping_symbol'].dropna())
unique_maping_maping_DB_SYMBOL	        = set(symbol_mapinr_df['maping_DB_SYMBOL'].dropna())
list_mapinr = [unique_maping_maping_currency_name, unique_maping_maping_currency_key_lr, unique_maping_maping_symbol, unique_maping_maping_DB_SYMBOL]

unique_crm_currency_name                = set(crm_symbol_df['currency_name'].dropna())
unique_crm_currency_key_lr              = set(crm_symbol_df['currency_key_lr'].dropna())
unique_crm_symbol                       = set(crm_symbol_df['_symbol'].dropna())
list_crm_symbol = [unique_crm_currency_name, unique_crm_currency_key_lr, unique_crm_symbol]

result = find_max_intersection(list_mapinr, list_crm_symbol)
for idx1, idx2, count in result:
    print(f"Множество {idx1} из list1 пересекается с множеством {idx2} из list2 на {count} элементов")

result = find_max_intersection(list_crm_symbol, list_mapinr)
for idx1, idx2, count in result:
    print(f"Множество {idx1} из list1 пересекается с множеством {idx2} из list2 на {count} элементов")

comparison_sets(unique_maping_maping_symbol, unique_crm_symbol)


In [10]:
# [ОБЯЗАТЕЛЕН] создаём таблицу соответствия символов CRM и МТ5 <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("Cоздаём таблицу ['crm_symbol_df'] соответствия символов CRM и МТ5")# ````````````````````````````````````````````````````````````````````````

crm_symbol_df["mapping"] = None                                         # колонка для символов на стороне МТ5

row_1 = '_symbol'                                                       # Колонка в ДФ с символами к поиску CRM;
row_3 = 'maping_symbol'                                                 # Колонка с символами к сопоставлению в symbol_mapinr_df;
row_2 = 'maping_DB_SYMBOL'                                              # Колонка с символами к открытию MT5;

for index, row in crm_symbol_df.iterrows():                             # Идем по строкам crm_symbol_df, перебираем символы CRM 
    currency_name = row[row_1]                                          # Фиксируем символ в crm_symbol_df
    for s_index, s_row in symbol_mapinr_df.iterrows():                  # Ищем соответствии в таблице сопоставления
        if currency_name == s_row[row_3]:
            if currency_name == "USD/AUD": print(currency_name, s_row[row_2])
            #print(f"{currency_name} = {s_row[row_2]}")
            crm_symbol_df.at[index, "mapping"] = s_row[row_2]           # Обновляем значение колонки "mapping" в оригинальном DataFrame
count_values = crm_symbol_df['mapping'].count()                         # Количество заполненных ячеек в crm_symbol_df["mapping"]

# заполняем пустые значениями значениями из CRM. т.к. некоторые названия в ЦРМ совпадают с реальными
crm_symbol_df['mapping'] = crm_symbol_df['mapping'].fillna(crm_symbol_df['_symbol']) # заполняем пустые значениями значениями из CRM. (Не точный)

print(f"Из [{len(crm_symbol_df)}] строк crm_symbol_df['mapping'], заполнено значениями из файла [{file_path}]: [{count_values}], \n",
      f"значениями из crm_symbol_df['_symbol']: [{len(crm_symbol_df) - count_values}]")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Определение символов без мапинга <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("\n Проверяем Все ли символы задействованные в тороговле имеют соответствие в таблице мапинга")# `````````````````````````````````````````
none_mapping_rows = crm_symbol_df[crm_symbol_df['mapping'].isnull()]                                  # ДФ символов CRM без мапинга
sorted_currency_list = sorted(list(set(none_mapping_rows['currency_name'])))                          # Множество символов CRM без мапинга
print(f"[{len(sorted_currency_list)}] cимволов CRM без мапинга: [{sorted_currency_list}]")
if len(sorted_currency_list) > 0:
    pd_set_option(f"DF[none_mapping_rows] Символы CRM без соответствия в файле [{file_path}]:", none_mapping_rows, 3)
    df_to_csv(none_mapping_rows, "Торговые инструменты без мапинга.csv")

# Определение торговых акаунтов с сделками по символам без мапинга <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("\n Определение торговых акаунтов с сделками по символам без мапинга")#``````````````````````````````````````````````````````````````````
filtered_df = statement_df[statement_df['symbol'].isin(sorted_currency_list)]                   # DF торговых операций по символам без мапинга
unique_account_ids = filtered_df['account_id'].unique().tolist()                                # Получаем уникальные account_id и сортируем
print(f"[{len(unique_account_ids)}] торговых акаунтов с сделками по символам без мапинга: [{unique_account_ids}]")
if len(unique_account_ids) > 0: 
    pd_set_option(f"акаунтов с сделками по символам без мапинга:", filtered_df, 3)

"""# Определение символов С мапингом <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
statement_symbol_df = crm_symbol_df[crm_symbol_df['mapping'].notnull()]
unique_statement_symbol_df = sorted(list(set(statement_symbol_df['_symbol'])))                    # Множество символов СТЕЙТМЕНТА с мапингом
print(f"\n [{len(unique_statement_symbol_df)}] cимволов СТЕЙТМЕНТА с мапингом: [{unique_statement_symbol_df}]")
pd_set_option(f"DF[statement_symbol_df] Символы СТЕЙТМЕНТА с соответствием в файле [{file_path}]:", statement_symbol_df, 3)"""

# Определение строк с совпадающими символами мапинга <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("\n Определение строк с совпадающими символами мапинга (разные id одинаковые текстовые имена)") # ``````````````````````````````````````````````````````````````````````````````````````````````
def non_unique_symbols_in_rows(df, column_name):
    non_unique_symbols = df[column_name].value_counts()                         # Группируем по колонке и подсчитываем количество вхождений каждого значения
    non_unique_symbols = non_unique_symbols[non_unique_symbols > 1].index       # Выбираем только те значения, которые встречаются более одного раза
    non_unique_df = df[df[column_name].isin(non_unique_symbols)]                # Отбираем строки, где '_' соответствует найденным неуникальным значениям
    non_unique_df = non_unique_df.sort_values(by=column_name, ascending=True)   # По возрастанию
    return non_unique_df
pd_set_option("non_unique_symbols_in_rows", non_unique_symbols_in_rows(crm_symbol_df, "mapping"), 10)

Cоздаём таблицу ['crm_symbol_df'] соответствия символов CRM и МТ5
Из [321] строк crm_symbol_df['mapping'], заполнено значениями из файла [files/Original/symbol_mapinr_.csv]: [287], 
 значениями из crm_symbol_df['_symbol']: [34]

 Проверяем Все ли символы задействованные в тороговле имеют соответствие в таблице мапинга
[0] cимволов CRM без мапинга: [[]]

 Определение торговых акаунтов с сделками по символам без мапинга
[0] торговых акаунтов с сделками по символам без мапинга: [[]]

 Определение строк с совпадающими символами мапинга (разные id одинаковые текстовые имена)
non_unique_symbols_in_rows


,currency_id,trading_group_id,parent_currency_id,currency_name,currency_key_lr,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_point_tech,symbol_multiply,symbol_calc_mode,symbol_contract_size,symbol_margin_initial,0_Open,0_Close,1_Open,1_Close,2_Open,2_Close,3_Open,3_Close,4_Open,4_Close,5_Open,5_Close,6_Open,6_Close,leverage,spread,swap_buy,swap_sell,minimum_change,commission,expiration,sell_last_value,buy_last_value,asset_id,asset_priority,asset_active,updated_at,mapping
182,448,1,448,Bank of America Corp,BAC,BAC,,USD,USD,2,0.01,0.010,100.0,2,100.0,0.1,0,0,870,1260,870,1260,870,1260,870,1260,870,1260,0,0,10,45,0,0,None,None,None,46.86000,46.86000,3,0,1,2025-01-27 22:27:39,BAC.N
87,127,1,127,#BOA,#BOA,BAC,,USD,USD,2,0.01,0.010,100.0,2,100.0,0.1,0,0,870,1260,870,1260,870,1260,870,1260,870,1260,0,0,10,45,0,0,None,None,None,32.17000,32.24000,3,0,0,2024-11-11 15:46:09,BAC.N
130,291,1,291,FRA40,FRA40,FRA40,,EUR,EUR,2,0.01,1.000,100.0,2,100.0,0.01,0,0,360,1200,360,1200,360,1200,360,1200,360,1200,0,0,100,6,0,0,None,None,None,7932.28000,7934.28000,4,0,1,2025-01-27 22:28:19,CAC40
48,68,1,68,CAC40,FCHI,CAC40,France 40 Cash Index,EUR,EUR,2,0.01,1.000,100.0,2,100.0,0.01,0,0,360,1200,360,1200,360,1200,360,1200,360,1200,0,0,100,60,0,0,None,None,None,7906.58000,7906.58000,4,0,1,2025-01-27 18:51:12,CAC40
135,308,1,308,CORNF,CORNF,CORNF,,USD,USD,2,0.01,1.000,100.0,2,100.0,0.1,0,0,30,1110,30,1110,30,1110,30,1110,30,1110,0,0,100,1,0,0,None,None,None,481.79000,482.95000,2,0,1,2025-01-27 21:19:37,CornX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,296,1,296,SPX500,SPX500,SPX500,,USD,USD,2,0.01,0.010,100.0,2,100.0,0.1,0,0,0,1439,0,1439,0,1439,0,1439,0,1439,0,0,100,1200,0,0,None,,None,6003.84000,6004.34000,4,0,1,2025-01-27 22:28:19,US500
128,288,1,288,USD/CNH,USDCNH,USDCNH,US Dollar vs Chinese Yuan Renminbi,CNH,USD,5,0.01,0.001,100000.0,0,100000.0,0,0,0,0,1440,0,1440,0,1440,0,1440,0,1440,0,0,400,30,0,0,None,None,None,7.25027,7.25082,1,0,1,2025-01-27 22:28:19,USDCNH
12,16,1,16,USD/CNY,USDCNY,USDCNY,,CNY,USD,5,0.01,0.001,100000.0,0,100000.0,0,0,0,0,1440,0,1440,0,1440,0,1440,0,1440,0,0,400,24,0,0,None,,None,7.24960,7.25160,1,0,1,2025-01-27 22:28:19,USDCNH
54,74,1,74,GOLD,GOLD,GOLD,Gold vs US-Dollar,USD,USD,2,0.01,0.001,1000.0,2,100.0,0.01,0,0,0,1439,0,1439,0,1439,0,1439,0,1439,0,0,100,4500,0,0,None,None,None,2739.63000,2739.90000,2,1,1,2025-01-27 22:28:19,XAUUSD


In [11]:
# [ОБЯЗАТЕЛЕН] Проверка соответствия настроек символов на стороне CRM и на стороне МТ5 <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
statement_symbol_df = crm_symbol_df[crm_symbol_df['mapping'].notnull()]

list_col_name = ['symbol_contract_size_mt5', 'available_group_clients']
add_list_col_name = True
new_position = 3
new_position_step = 0
customers_accounts_crm_mt5_df = move_column(statement_symbol_df, list_col_name, add_list_col_name, new_position, new_position_step)

list_col_name = list_col_name + ['currency_name', 'mapping', 'symbol_contract_size']
add_list_col_name = False
new_position = 0
new_position_step = 1
customers_accounts_crm_mt5_df = move_column(statement_symbol_df, list_col_name, add_list_col_name, new_position, new_position_step)

new_df = pd.DataFrame(columns=customers_accounts_crm_mt5_df.columns)

manager = manager_connect()
if manager:
    print("Успешное соеденение MT5 manager")
    group_request_result = manager.GroupRequest("demo\\technical_dev\\migration_zero_usd")
    
    for index, row in customers_accounts_crm_mt5_df.iterrows():                         # Проход по строкам DataFrame сверху вниз
        symbol_mapping = row['mapping']

        symbol_request_result = manager.SymbolRequest(symbol_mapping)
        if symbol_request_result:
            symbol_exist = manager.SymbolExist(symbol_request_result, group_request_result)

            if manager.SymbolRequest(symbol_mapping):
                customers_accounts_crm_mt5_df.at[index, 'available_group_clients'] = True
            else: customers_accounts_crm_mt5_df.at[index, 'available_group_clients'] = False

            contract_size = symbol_request_result.ContractSize 
            customers_accounts_crm_mt5_df.at[index, 'symbol_contract_size_mt5'] = contract_size
            if row['symbol_contract_size'] != contract_size:
                print(f"ERROR: symbol_mapping = {symbol_mapping}, CRM contract_size = {row['symbol_contract_size']}, MT5 = {contract_size}")

            #print(f"symbol_mapping = {symbol_mapping}, symbol_exist = {symbol_exist}, contract_size = {contract_size}")
        else:
            print("ERROR: символ не найден", symbol_mapping)
            new_df = pd.concat([new_df, pd.DataFrame([row])], ignore_index=True)

print(manager_disconnect())

pd_set_option(" \n Таблица Символов не найденных в МТ5", new_df, 3)

"""# Оставляем только строки, где значения в 'symbol_digits' не являются цифровыми
non_digit_df = customers_accounts_crm_mt5_df[~customers_accounts_crm_mt5_df['symbol_digits'].astype(str).str.isdigit()]
pd_set_option(" \n Таблица Символов из CRM ", non_digit_df, 10)

# Фильтруем строки, где значение в колонке 'symbol_digits' равно NaN
nan_rows_df = customers_accounts_crm_mt5_df[pd.isna(customers_accounts_crm_mt5_df['symbol_digits'])]
pd_set_option(" \n Таблица Символов из CRM ", nan_rows_df, 10)"""

pd_set_option(" \n Таблица Символов из CRM ", customers_accounts_crm_mt5_df, 3)

 
 Функция добавления новых колонок с пустыми значениями и изменения положения этих колонок  
 list_col_name = ['symbol_contract_size_mt5', 'available_group_clients'], new_position = 3
Перемещена колонка 'symbol_contract_size_mt5' на позицию 3
Перемещена колонка 'available_group_clients' на позицию 3
 
 Функция добавления новых колонок с пустыми значениями и изменения положения этих колонок  
 list_col_name = ['symbol_contract_size_mt5', 'available_group_clients', 'currency_name', 'mapping', 'symbol_contract_size'], new_position = 0
Перемещена колонка 'symbol_contract_size_mt5' на позицию 0
Перемещена колонка 'available_group_clients' на позицию 1
Перемещена колонка 'currency_name' на позицию 2
Перемещена колонка 'mapping' на позицию 3
Перемещена колонка 'symbol_contract_size' на позицию 4
import.py manager =  <MT5Manager.ManagerAPI object at 0x000002EFE16AEFB0>


import.py MT5manager connect: True
Успешное соеденение MT5 manager
import.py manager.Disconnect() True
None
 
 Таблица Символов не найденных в МТ5


,symbol_contract_size_mt5,available_group_clients,currency_name,mapping,symbol_contract_size,currency_id,trading_group_id,parent_currency_id,currency_key_lr,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_point_tech,symbol_multiply,symbol_calc_mode,symbol_margin_initial,0_Open,0_Close,1_Open,1_Close,2_Open,2_Close,3_Open,3_Close,4_Open,4_Close,5_Open,5_Close,6_Open,6_Close,leverage,spread,swap_buy,swap_sell,minimum_change,commission,expiration,sell_last_value,buy_last_value,asset_id,asset_priority,asset_active,updated_at


 
 Таблица Символов из CRM 


,symbol_contract_size_mt5,available_group_clients,currency_name,mapping,symbol_contract_size,currency_id,trading_group_id,parent_currency_id,currency_key_lr,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_point_tech,symbol_multiply,symbol_calc_mode,symbol_margin_initial,0_Open,0_Close,1_Open,1_Close,2_Open,2_Close,3_Open,3_Close,4_Open,4_Close,5_Open,5_Close,6_Open,6_Close,leverage,spread,swap_buy,swap_sell,minimum_change,commission,expiration,sell_last_value,buy_last_value,asset_id,asset_priority,asset_active,updated_at
0,100000.0,True,EUR/USD,EURUSD,100000.0,1,1,1,EURUSD,EURUSD,Euro vs US Dollar,USD,EUR,5,0.01,0.0001,100000.0,0,0,0,0,0,1440,0,1440,0,1440,0,1440,0,1440,0,0,400,9,0,0,None,None,None,1.04911,1.04917,1,0,1,2025-01-27 22:28:19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,10.0,True,iShares TIPS Bond ETF,TIP.ETF,10.0,974,1,974,TIP,TIP,,USD,USD,3,0.01,0.0100,100.0,2,0.1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,10,1,1,0,None,None,None,107.92000,107.92000,3,0,1,2025-01-27 22:25:39


In [ ]:
# Сохраняем DataFrame c символами отсутствующими в МАПИНГЕ в CSV файл <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
csv_file_path = "files/symbols_to_create.csv"
df_to_csv(none_mapping_rows, csv_file_path)

In [ ]:
# Сохраняем DataFrame c символами CRM и данными о МАПИНГЕ в CSV файл <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
csv_file_path = "files/crm_symbol_df_map.csv"
df_to_csv(crm_symbol_df, csv_file_path)

In [12]:
# [ОБЯЗАТЕЛЕН] Объединение Данных по торговым транзакциям и спецификаций инструментов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
unique_sorted_df = customers_accounts_crm_mt5_df
#pd_set_option("[unique_sorted_df] символы CRM ", unique_sorted_df, 3)
deals_and_symbols_df = statement_df.merge(unique_sorted_df, how='left', left_on='symbol', right_on='currency_name') # Объединение DataFrame на основе 'symbol' и 'currency_name'
print(f"len(statement_df): {len(statement_df)}; len(unique_sorted_df): {len(unique_sorted_df)}; len(deals_and_symbols_df): {len(deals_and_symbols_df)}")
pd_set_option("[unique_sorted_df] Объединение DataFrame на основе ['Symbol'] и ['currency_name']", deals_and_symbols_df, 3)

len(statement_df): 133223; len(unique_sorted_df): 321; len(deals_and_symbols_df): 133223
[unique_sorted_df] Объединение DataFrame на основе ['Symbol'] и ['currency_name']


,order_id,ip,lead_id,account_id,currency_id_x,symbol,position_id,re_open,volume_lots,leverage_x,margin,spread_x,command,open_price,current_rate,close_price,open_time,close_time,close_at_profit,close_at_profit_type,close_at_loss,close_at_loss_type,start_at_price,profit,swap,sync_date,execute_in_future,execute_in_future_params,execute_in_future_processed,symbol_contract_size_mt5,available_group_clients,currency_name,mapping,symbol_contract_size,currency_id_y,trading_group_id,parent_currency_id,currency_key_lr,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_point_tech,symbol_multiply,symbol_calc_mode,symbol_margin_initial,0_Open,0_Close,1_Open,1_Close,2_Open,2_Close,3_Open,3_Close,4_Open,4_Close,5_Open,5_Close,6_Open,6_Close,leverage_y,spread_y,swap_buy,swap_sell,minimum_change,commission,expiration,sell_last_value,buy_last_value,asset_id,asset_priority,asset_active,updated_at
0,260,95.217.166.120,163014,163012,151,_BAYER,3,0,1.00,10,5.60,-0.01,0,51.405,50.685,50.685,2023-07-03 12:07:42,2023-07-04 10:00:21,NaN,2,NaN,2,NaN,-0.79,-2.83,2023-07-04 10:00:20,None,None,0,100.0,True,_BAYER,BAYN.XE,100.0,151,1,151,_BAYER,_BAYER,,EUR,EUR,3,0.01,0.01,100.0,2,0.1,0,0,420,990,420,990,420,990,420,990,420,990,0,0,10,45,0,0,None,None,None,21.415,21.515,6,0,1,2025-01-27 18:29:20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133222,209249,2a02:6680:110d:39af:afb4:ba49:7ed6:d29f,267250,265987,101,PLATINUM,1,0,0.01,100,9.73,-2.80,0,973.000,963.100,NaN,2025-01-24 23:56:29,NaT,NaN,2,NaN,2,NaN,-12.70,NaN,2025-01-27 22:28:26,None,None,0,100.0,True,PLATINUM,XPTUSD,100.0,101,1,101,PLATINUM,PLATINUM,,USD,USD,2,0.01,0.10,100.0,2,0.05,0,0,0,1439,0,1439,0,1439,0,1439,0,1439,0,0,100,28,0,0,None,None,None,960.200,963.100,2,0,1,2025-01-27 22:28:19


In [ ]:
columns_to_check = ['symbol_digits']    # Список колонок для проверки
for column in columns_to_check:         # Преобразуем значения в числовой формат, при этом ошибки не заменяются на NaN
    # Мы не изменяем данные в колонке, а просто проверяем, можно ли их преобразовать
    deals_and_symbols_df[column + '_check'] = pd.to_numeric(deals_and_symbols_df[column], errors='coerce')

# Отбираем строки, где не удалось преобразовать в числовой формат (т.е. где значение NaN)
invalid_rows_df = deals_and_symbols_df[deals_and_symbols_df[[col + '_check' for col in columns_to_check]].isna().any(axis=1)]
unique_account_ids = invalid_rows_df['account_id'].unique()
print(unique_account_ids)
pd_set_option("invalid_rows_df", invalid_rows_df, 3)

pd.set_option('display.max_rows', None)
print("\n", deals_and_symbols_df["symbol_digits"].value_counts(dropna=False), "\n")

In [ ]:
"""filtr_df = statement_format_df[statement_format_df["position_id"] == 115419]
pd_set_option("statement_df[""] ==", filtr_df, 3)"""

In [ ]:
# [ОБЯЗАТЕЛЕН] создание служебных колонок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# `````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````` 

columns = deals_and_symbols_df.columns.tolist()
deals_and_symbols_df["profit_crm"] = deals_and_symbols_df["profit"] - deals_and_symbols_df["spread_x"]                                 # Итог по позиции в CRM

# Создаём Колонку с итогом в ПУНКТАХ по ПОЗИЦИИ (ЗАКРЫТОЙ / ОТКРЫТОЙ) <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< 
deals_and_symbols_df["point_profit"] = np.where(deals_and_symbols_df["position_id"] == 3,#deals_and_symbols_df["close_price"] > 0,
                                     deals_and_symbols_df["close_price"] - deals_and_symbols_df["open_price"],
                                     deals_and_symbols_df['current_rate'] - deals_and_symbols_df["open_price"])

# Создаём Колонку с коэффициентом пересчёта из валюты прибыли СДЕЛКИ в БАЛАНСОВУЮ валюту <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
deals_and_symbols_df["profit_right"] = deals_and_symbols_df["point_profit"] * deals_and_symbols_df["volume_lots"] * deals_and_symbols_df["symbol_contract_size"]

# Меняем порядок колонок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
list_col_name = ['profit_crm', 'point_profit','profit_right', 'rate_profit']
add_list_col_name = False
new_position = 1
new_position_step = 1
deals_and_symbols_df = move_column(deals_and_symbols_df, list_col_name, add_list_col_name, new_position, new_position_step)

pd_set_option("Создание служебных колонок: deals_and_symbols_df[profit_crm],[point_profit],[profit_right],[volume_lots],[symbol_contract_size]", deals_and_symbols_df, 10)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Пользовательская функция для расчета 'rate_profit'<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def calculate_rate_profit(row):
    if row['profit_right'] is None or row['profit_right'] == 0: return 1
    else: return round(row['profit_crm'] / row['profit_right'], 8)
deals_and_symbols_df['rate_profit'] = deals_and_symbols_df.apply(calculate_rate_profit, axis=1) # Применение функции к каждой строке DataFrame
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>


#deals_and_symbols_df["rate_profit"] = round(deals_and_symbols_df["profit_crm"] / deals_and_symbols_df["profit_right"], 8)  была изначально

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>order_id</th>
      <th>profit_crm</th>
      <th>point_profit</th>
      <th>profit_right</th>
      <th>ip</th>
      <th>lead_id</th>
      <th>account_id</th>
      <th>currency_id_x</th>
      <th>symbol</th>
      <th>position_id</th>
      <th>re_open</th>
      <th>volume_lots</th>
      <th>leverage_x</th>
      <th>margin</th>
      <th>spread_x</th>
      <th>command</th>
      <th>open_price</th>
      <th>current_rate</th>
      <th>close_price</th>
      <th>open_time</th>
      <th>close_time</th>
      <th>close_at_profit</th>
      <th>close_at_profit_type</th>
      <th>close_at_loss</th>
      <th>close_at_loss_type</th>
      <th>start_at_price</th>
      <th>profit</th>
      <th>swap</th>
      <th>sync_date</th>
      <th>execute_in_future</th>
      <th>execute_in_future_params</th>
      <th>execute_in_future_processed</th>
      <th>symbol_contract_size_mt5</th>
      <th>available_group_clients</th>
      <th>currency_name</th>
      <th>mapping</th>
      <th>symbol_contract_size</th>
      <th>currency_id_y</th>
      <th>trading_group_id</th>
      <th>parent_currency_id</th>
      <th>currency_key_lr</th>
      <th>_symbol</th>
      <th>symbol_description</th>
      <th>symbol_profit</th>
      <th>symbol_margin</th>
      <th>symbol_digits</th>
      <th>symbol_point</th>
      <th>symbol_point_tech</th>
      <th>symbol_multiply</th>
      <th>symbol_calc_mode</th>
      <th>symbol_margin_initial</th>
      <th>0_Open</th>
      <th>0_Close</th>
      <th>1_Open</th>
      <th>1_Close</th>
      <th>2_Open</th>
      <th>2_Close</th>
      <th>3_Open</th>
      <th>3_Close</th>
      <th>4_Open</th>
      <th>4_Close</th>
      <th>5_Open</th>
      <th>5_Close</th>
      <th>6_Open</th>
      <th>6_Close</th>
      <th>leverage_y</th>
      <th>spread_y</th>
      <th>swap_buy</th>
      <th>swap_sell</th>
      <th>minimum_change</th>
      <th>commission</th>
      <th>expiration</th>
      <th>sell_last_value</th>
      <th>buy_last_value</th>
      <th>asset_id</th>
      <th>asset_priority</th>
      <th>asset_active</th>
      <th>updated_at</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>260</td>
      <td>-0.78</td>
      <td>-0.72000</td>
      <td>-7.200000e+01</td>
      <td>95.217.166.120</td>
      <td>163014</td>
      <td>163012</td>
      <td>151</td>
      <td>_BAYER</td>
      <td>3</td>
      <td>0</td>
      <td>1.00</td>
      <td>10</td>
      <td>5.60</td>
      <td>-0.01</td>
      <td>0</td>
      <td>51.40500</td>
      <td>50.68500</td>
      <td>50.68500</td>
      <td>2023-07-03 12:07:42</td>
      <td>2023-07-04 10:00:21</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>-0.79</td>
      <td>-2.83</td>
      <td>2023-07-04 10:00:20</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>100.0</td>
      <td>True</td>
      <td>_BAYER</td>
      <td>BAYN.XE</td>
      <td>100.0</td>
      <td>151</td>
      <td>1</td>
      <td>151</td>
      <td>_BAYER</td>
      <td>_BAYER</td>
      <td></td>
      <td>EUR</td>
      <td>EUR</td>
      <td>3</td>
      <td>0.01</td>
      <td>0.0100</td>
      <td>100.0</td>
      <td>2</td>
      <td>0.1</td>
      <td>0</td>
      <td>0</td>
      <td>420</td>
      <td>990</td>
      <td>420</td>
      <td>990</td>
      <td>420</td>
      <td>990</td>
      <td>420</td>
      <td>990</td>
      <td>420</td>
      <td>990</td>
      <td>0</td>
      <td>0</td>
      <td>10</td>
      <td>45</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>21.41500</td>
      <td>21.51500</td>
      <td>6</td>
      <td>0</td>
      <td>1</td>
      <td>2025-01-27 18:29:20</td>
    </tr>
    <tr>
      <th>1</th>
      <td>261</td>
      <td>7.01</td>
      <td>-7.00571</td>
      <td>-7.005710e+01</td>
      <td>95.217.166.120</td>
      <td>163014</td>
      <td>163012</td>
      <td>13</td>
      <td>ETH</td>
      <td>3</td>
      <td>0</td>
      <td>1.00</td>
      <td>5</td>
      <td>392.09</td>
      <td>-16.00</td>
      <td>1</td>
      <td>1960.66951</td>
      <td>1953.66380</td>
      <td>1953.66380</td>
      <td>2023-07-03 12:09:07</td>
      <td>2023-07-04 10:00:10</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>-8.99</td>
      <td>-0.02</td>
      <td>2023-07-04 10:00:09</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>10.0</td>
      <td>True</td>
      <td>ETH</td>
      <td>ETHUSD</td>
      <td>10.0</td>
      <td>13</td>
      <td>1</td>
      <td>13</td>
      <td>ETH</td>
      <td>ETHUSD</td>
      <td>Ethereum vs. USD</td>
      <td>USD</td>
      <td>USD</td>
      <td>2</td>
      <td>0.01</td>
      <td>1.0000</td>
      <td>100.0</td>
      <td>2</td>
      <td>0.5</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>400</td>
      <td>90</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>3909.93970</td>
      <td>3909.93970</td>
      <td>5</td>
      <td>0</td>
      <td>0</td>
      <td>2024-12-13 12:00:20</td>
    </tr>
    <tr>
      <th>2</th>
      <td>262</td>
      <td>190.00</td>
      <td>0.19000</td>
      <td>1.900000e+02</td>
      <td>95.217.166.120</td>
      <td>163014</td>
      <td>163012</td>
      <td>78</td>
      <td>BRENT_OIL</td>
      <td>3</td>
      <td>0</td>
      <td>1.00</td>
      <td>20</td>
      <td>3811.00</td>
      <td>0.00</td>
      <td>0</td>
      <td>76.22000</td>
      <td>76.41000</td>
      <td>76.41000</td>
      <td>2023-07-03 12:10:22</td>
      <td>2023-07-03 12:14:28</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>190.00</td>
      <td>NaN</td>
      <td>2023-07-03 12:14:26</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>1000.0</td>
      <td>True</td>
      <td>BRENT_OIL</td>
      <td>BRTSPOT</td>
      <td>1000.0</td>
      <td>78</td>
      <td>1</td>
      <td>78</td>
      <td>BRENT_OIL</td>
      <td>BRENT_OIL</td>
      <td></td>
      <td>USD</td>
      <td>USD</td>
      <td>2</td>
      <td>0.01</td>
      <td>0.0010</td>
      <td>1000.0</td>
      <td>2</td>
      <td>0.01</td>
      <td>0</td>
      <td>0</td>
      <td>1</td>
      <td>1439</td>
      <td>1</td>
      <td>1439</td>
      <td>1</td>
      <td>1439</td>
      <td>1</td>
      <td>1439</td>
      <td>1</td>
      <td>1439</td>
      <td>0</td>
      <td>0</td>
      <td>100</td>
      <td>135</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>76.20000</td>
      <td>76.23000</td>
      <td>2</td>
      <td>1</td>
      <td>1</td>
      <td>2025-01-27 22:26:59</td>
    </tr>
    <tr>
      <th>3</th>
      <td>724</td>
      <td>2.52</td>
      <td>0.00084</td>
      <td>2.520000e+00</td>
      <td>176.230.45.6</td>
      <td>223159</td>
      <td>223157</td>
      <td>1</td>
      <td>EUR/USD</td>
      <td>3</td>
      <td>0</td>
      <td>0.03</td>
      <td>400</td>
      <td>8.16</td>
      <td>-0.90</td>
      <td>0</td>
      <td>1.08810</td>
      <td>1.08894</td>
      <td>1.08894</td>
      <td>2023-07-05 15:17:12</td>
      <td>2023-07-05 15:32:04</td>
      <td>1.5</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>1.62</td>
      <td>NaN</td>
      <td>2023-07-05 15:32:04</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>100000.0</td>
      <td>True</td>
      <td>EUR/USD</td>
      <td>EURUSD</td>
      <td>100000.0</td>
      <td>1</td>
      <td>1</td>
      <td>1</td>
      <td>EURUSD</td>
      <td>EURUSD</td>
      <td>Euro vs US Dollar</td>
      <td>USD</td>
      <td>EUR</td>
      <td>5</td>
      <td>0.01</td>
      <td>0.0001</td>
      <td>100000.0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>0</td>
      <td>400</td>
      <td>9</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>1.04911</td>
      <td>1.04917</td>
      <td>1</td>
      <td>0</td>
      <td>1</td>
      <td>2025-01-27 22:28:19</td>
    </tr>
    <tr>
      <th>4</th>
      <td>725</td>
      <td>-5.34</td>
      <td>0.00178</td>
      <td>5.340000e+00</td>
      <td>176.230.45.6</td>
      <td>223159</td>
      <td>223157</td>
      <td>2</td>
      <td>GBP/USD</td>
      <td>3</td>
      <td>0</td>
      <td>0.03</td>
      <td>400</td>
      <td>9.53</td>
      <td>9.00</td>
      <td>1</td>
      <td>1.26997</td>
      <td>1.27175</td>
      <td>1.27175</td>
      <td>2023-07-05 15:21:49</td>
      <td>2023-07-06 17:03:25</td>
      <td>1.5</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>3.66</td>
      <td>-0.90</td>
      <td>2023-07-06 17:03:25</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>100000.0</td>
      <td>True</td>
      <td>GBP/USD</td>
      <td>GBPUSD</td>
      <td>100000.0</td>
      <td>2</td>
      <td>1</td>
      <td>2</td>
      <td>GBPUSD</td>
      <td>GBPUSD</td>
      <td>Great Britain Pound vs US Dollar</td>
      <td>USD</td>
      <td>GBP</td>
      <td>5</td>
      <td>0.01</td>
      <td>0.0010</td>
      <td>100000.0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>0</td>
      <td>400</td>
      <td>3</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>1.24899</td>
      <td>1.24912</td>
      <td>1</td>
      <td>0</td>
      <td>1</td>
      <td>2025-01-27 22:28:19</td>
    </tr>
    <tr>
      <th>...</th>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
    </tr>
    <tr>
      <th>133218</th>
      <td>209245</td>
      <td>-885.75</td>
      <td>-29.52500</td>
      <td>-8.857500e+02</td>
      <td>79.183.217.28</td>
      <td>265651</td>
      <td>264387</td>
      <td>570</td>
      <td>Coinbase Global, Inc.</td>
      <td>1</td>
      <td>0</td>
      <td>0.30</td>
      <td>10</td>
      <td>895.92</td>
      <td>-13.50</td>
      <td>0</td>
      <td>298.84500</td>
      <td>269.32000</td>
      <td>NaN</td>
      <td>2025-01-24 22:57:27</td>
      <td>NaT</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>-899.25</td>
      <td>NaN</td>
      <td>2025-01-27 22:27:51</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>100.0</td>
      <td>True</td>
      <td>Coinbase Global, Inc.</td>
      <td>COIN.OQ</td>
      <td>100.0</td>
      <td>570</td>
      <td>1</td>
      <td>570</td>
      <td>COIN</td>
      <td>COIN</td>
      <td></td>
      <td>USD</td>
      <td>USD</td>
      <td>2</td>
      <td>0.01</td>
      <td>0.0100</td>
      <td>100.0</td>
      <td>2</td>
      <td>0.1</td>
      <td>0</td>
      <td>0</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>0</td>
      <td>0</td>
      <td>10</td>
      <td>45</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>268.92000</td>
      <td>268.92000</td>
      <td>3</td>
      <td>0</td>
      <td>1</td>
      <td>2025-01-27 22:27:39</td>
    </tr>
    <tr>
      <th>133219</th>
      <td>209246</td>
      <td>-214.55</td>
      <td>-21.45500</td>
      <td>-2.145500e+02</td>
      <td>79.183.217.28</td>
      <td>265651</td>
      <td>264387</td>
      <td>441</td>
      <td>NVIDIA Corp</td>
      <td>3</td>
      <td>0</td>
      <td>0.10</td>
      <td>10</td>
      <td>142.49</td>
      <td>-4.50</td>
      <td>0</td>
      <td>142.60500</td>
      <td>121.15000</td>
      <td>121.15000</td>
      <td>2025-01-24 23:00:03</td>
      <td>2025-01-27 18:08:41</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>-219.05</td>
      <td>NaN</td>
      <td>2025-01-27 18:08:31</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>100.0</td>
      <td>True</td>
      <td>NVIDIA Corp</td>
      <td>NVDA.OQ</td>
      <td>100.0</td>
      <td>441</td>
      <td>1</td>
      <td>441</td>
      <td>NVDA</td>
      <td>NVDA</td>
      <td></td>
      <td>USD</td>
      <td>USD</td>
      <td>2</td>
      <td>0.01</td>
      <td>0.0100</td>
      <td>100.0</td>
      <td>2</td>
      <td>0.1</td>
      <td>0</td>
      <td>0</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>0</td>
      <td>0</td>
      <td>10</td>
      <td>45</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>117.84000</td>
      <td>117.84000</td>
      <td>3</td>
      <td>0</td>
      <td>1</td>
      <td>2025-01-27 22:28:19</td>
    </tr>
    <tr>
      <th>133220</th>
      <td>209247</td>
      <td>-836.40</td>
      <td>-27.88000</td>
      <td>-8.364000e+02</td>
      <td>2600:1702:5c60:8240:e0e4:8378:b30:d39d</td>
      <td>238353</td>
      <td>238367</td>
      <td>570</td>
      <td>Coinbase Global, Inc.</td>
      <td>1</td>
      <td>0</td>
      <td>0.30</td>
      <td>10</td>
      <td>894.30</td>
      <td>-13.50</td>
      <td>0</td>
      <td>297.20000</td>
      <td>269.32000</td>
      <td>NaN</td>
      <td>2025-01-24 23:00:25</td>
      <td>NaT</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>-849.90</td>
      <td>NaN</td>
      <td>2025-01-27 22:27:51</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>100.0</td>
      <td>True</td>
      <td>Coinbase Global, Inc.</td>
      <td>COIN.OQ</td>
      <td>100.0</td>
      <td>570</td>
      <td>1</td>
      <td>570</td>
      <td>COIN</td>
      <td>COIN</td>
      <td></td>
      <td>USD</td>
      <td>USD</td>
      <td>2</td>
      <td>0.01</td>
      <td>0.0100</td>
      <td>100.0</td>
      <td>2</td>
      <td>0.1</td>
      <td>0</td>
      <td>0</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>870</td>
      <td>1260</td>
      <td>0</td>
      <td>0</td>
      <td>10</td>
      <td>45</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>268.92000</td>
      <td>268.92000</td>
      <td>3</td>
      <td>0</td>
      <td>1</td>
      <td>2025-01-27 22:27:39</td>
    </tr>
    <tr>
      <th>133221</th>
      <td>209248</td>
      <td>8423.00</td>
      <td>3.28000</td>
      <td>3.280000e+06</td>
      <td>74.51.231.63</td>
      <td>238541</td>
      <td>238555</td>
      <td>51</td>
      <td>CHF/HUF</td>
      <td>1</td>
      <td>0</td>
      <td>10.00</td>
      <td>400</td>
      <td>2759.56</td>
      <td>-1133.15</td>
      <td>0</td>
      <td>428.91000</td>
      <td>432.19000</td>
      <td>NaN</td>
      <td>2025-01-24 23:10:05</td>
      <td>NaT</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>7289.85</td>
      <td>NaN</td>
      <td>2025-01-27 22:28:27</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>100000.0</td>
      <td>True</td>
      <td>CHF/HUF</td>
      <td>CHFHUF</td>
      <td>100000.0</td>
      <td>51</td>
      <td>1</td>
      <td>51</td>
      <td>CHFHUF</td>
      <td>CHFHUF</td>
      <td></td>
      <td>HUF</td>
      <td>CHF</td>
      <td>5</td>
      <td>0.01</td>
      <td>0.0010</td>
      <td>100000.0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>0</td>
      <td>400</td>
      <td>440</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>431.64000</td>
      <td>432.11000</td>
      <td>1</td>
      <td>0</td>
      <td>1</td>
      <td>2025-01-27 22:28:19</td>
    </tr>
    <tr>
      <th>133222</th>
      <td>209249</td>
      <td>-9.90</td>
      <td>-9.90000</td>
      <td>-9.900000e+00</td>
      <td>2a02:6680:110d:39af:afb4:ba49:7ed6:d29f</td>
      <td>267250</td>
      <td>265987</td>
      <td>101</td>
      <td>PLATINUM</td>
      <td>1</td>
      <td>0</td>
      <td>0.01</td>
      <td>100</td>
      <td>9.73</td>
      <td>-2.80</td>
      <td>0</td>
      <td>973.00000</td>
      <td>963.10000</td>
      <td>NaN</td>
      <td>2025-01-24 23:56:29</td>
      <td>NaT</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>2</td>
      <td>NaN</td>
      <td>-12.70</td>
      <td>NaN</td>
      <td>2025-01-27 22:28:26</td>
      <td>None</td>
      <td>None</td>
      <td>0</td>
      <td>100.0</td>
      <td>True</td>
      <td>PLATINUM</td>
      <td>XPTUSD</td>
      <td>100.0</td>
      <td>101</td>
      <td>1</td>
      <td>101</td>
      <td>PLATINUM</td>
      <td>PLATINUM</td>
      <td></td>
      <td>USD</td>
      <td>USD</td>
      <td>2</td>
      <td>0.01</td>
      <td>0.1000</td>
      <td>100.0</td>
      <td>2</td>
      <td>0.05</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>0</td>
      <td>100</td>
      <td>28</td>
      <td>0</td>
      <td>0</td>
      <td>None</td>
      <td>None</td>
      <td>None</td>
      <td>960.20000</td>
      <td>963.10000</td>
      <td>2</td>
      <td>0</td>
      <td>1</td>
      <td>2025-01-27 22:28:19</td>
    </tr>
  </tbody>
</table>
<p>133223 rows × 78 columns</p>
</div>

In [14]:
# [ОБЯЗАТЕЛЕН]  форматирование <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# `````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````` 

statement_format_df = statement_format(deals_and_symbols_df)                                                          # Используем функцию для форматирования ДФ

csv_file_path = "statement_format_df.csv"
df_to_csv(statement_format_df, csv_file_path)

columns = statement_format_df.columns.tolist()

# рассчитываем финансовый результат, по сделке, в пунктах <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< 
statement_format_df["point_profit"] = (np.where(statement_format_df["position_id"] == 3,                                            # если CLOSE
                                               np.where(statement_format_df["command"] == 0,                                        # если сделка
                                                        statement_format_df["close_price"] - statement_format_df["open_price"],     # если сделка BUY и сделка CLOSE 
                                                        statement_format_df["open_price"] - statement_format_df["close_price"]),    # если сделка SELL и сделка CLOSE
                                                            np.where(statement_format_df["command"] == 1,                                       # если POSITIONS
                                                                    statement_format_df["open_price"] - statement_format_df["current_rate"],    # если POSITIONS SELL
                                                                    statement_format_df["current_rate"] - statement_format_df['open_price'])))  # если POSITIONS BUY
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

statement_format_df["profit_right"] = statement_format_df["point_profit"] * statement_format_df["volume_lots"] * statement_format_df["symbol_contract_size"]


# Пользовательская функция для расчета 'rate_profit'<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def calculate_rate_profit(row):
    if row['profit_right'] is None or row['profit_right'] == 0: return 1
    else: return round(row['profit_crm'] / row['profit_right'], 8)
statement_format_df['rate_profit'] = statement_format_df.apply(calculate_rate_profit, axis=1) # Применение функции к каждой строке DataFrame
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
#statement_format_df["rate_profit"]  = abs(round(statement_format_df["profit_crm"] / statement_format_df["profit_right"], 8))



statement_format_df["rate_profit"].fillna(1, inplace=True)              # заполняет все NaN значения в колонке rate_profit значением 1 и делает это "на месте," 
statement_format_df["comm"] = np.where(statement_format_df["command"] > 0, -1, 1)


# Рассчитываем цену закрытия в закрытых позициях без цены закрытия <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````

# Cоздание служебных колонок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
new_columns_pos_n = 1                                                                       # позиция с которой начинается добавление колонки
new_columns_list = ['old_close_price']                                                      # список добавляемых колонок
add_list_col_name = True
statement_format_df = move_column(statement_format_df, new_columns_list, add_list_col_name, new_columns_pos_n) # Перемещаем колонки / Добавляем новые колонки,

# переместим существующие колонки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
new_columns_pos_n = 1                                                                       # позиция колонки
new_columns_list = ['close_price']                                                          # список колонок
add_list_col_name = False
statement_format_df = move_column(statement_format_df, new_columns_list, add_list_col_name, new_columns_pos_n) # Перемещаем колонки / Добавляем новые колонки,

columns = list(statement_format_df.columns)
errors_close_price_closed_deals_df = pd.DataFrame(columns=columns)  # Для ошибок связанных с ценой открытия
try:
    for index, row in statement_format_df.iterrows():  # Перебираем строки DataFrame
        close_price = row.get("close_price", None)
        position_id = row.get("position_id", None)
        try:
            if ((pd.isna(close_price) or close_price == 0 or not isinstance(close_price, (int, float))) and (position_id == 3)):
                statement_format_df.at[index, 'old_close_price'] = close_price
                profit                                           = row["profit"] - row["spread_x"]
                lots                                             = row["volume_lots"]
                contract_size                                    = row["symbol_contract_size"]
                price_o                                          = row["open_price"]
                #print(f"row[symbol_digits] = {row["symbol_digits"]}")
                symbol_digits = row["symbol_digits"]
                price_c                                          = round((profit / (lots * contract_size)) * row["comm"] + price_o, int(symbol_digits))
                statement_format_df.at[index, "close_price"]     = price_c

                errors_close_price_closed_deals_df = pd.concat([errors_close_price_closed_deals_df, statement_format_df.loc[[index]]], ignore_index=True)
        except Exception as e:
            print(f"Error processing row at index {index}: {e}")
            error_rows_df = pd.concat([error_rows_df, statement_format_df.loc[[index]]], ignore_index=True)

except Exception as e:
    print("\nERROR в рассчете цен закрытия в закрытых позициях без цены закрытия:", e)


pd_set_option("\n Закрытые сделки с ОТСУТСТВУЮЩЕЙ ценой закрытия [errors_close_price_closed_deals_df] \n Цена закрытия заменена расчётной, указана старая цена ['old_close_price']", 
              errors_close_price_closed_deals_df, 10)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>       

# Создаем DataFrame с ошибочными сделками <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
errors_close_price_closed_deals_df = statement_format_df[(statement_format_df['position_id'] == 3) &
                                                         (pd.isna(statement_format_df['close_price']) | (statement_format_df['close_price'] == 0) |
                                                          ~statement_format_df['close_price'].apply(lambda x: isinstance(x, (int, float))))].copy()
pd_set_option(" \n Закрытые сделки с ошибкми в цене закрытия [statement_format_df]", errors_close_price_closed_deals_df, 5)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Проверяем соответствие расчётной прибыли с прибылью в CRM <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
statement_format_df["calc_profit_2"] = (statement_format_df["point_profit"]
                                      * (statement_format_df["volume_lots"] * statement_format_df["symbol_contract_size"])
                                      #* statement_format_df["comm"]
                                      * statement_format_df["rate_profit"])                                                                                      
statement_format_df["calc_crm_profit_check_2"] = statement_format_df["calc_profit_2"] - statement_format_df["profit_crm"]
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# определяем сделки по которым у символов нет размера контрактов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<https://fintech-area.slack.com/archives/G0165A4V6M7/p1730392923722159?thread_ts=1730248723.527459&cid=G0165A4V6M7
pd.options.display.float_format = '{:,.5f}'.format                                                                       # отображение до 5 знаков после запятой

import math
def round_down_to_significant_digit(x):                 
    if x == 0: return 0
    else:
        order_of_magnitude = round(math.log10(abs(x)))                                                          # Находим порядок числа (разрядность), используя округление
        rounded_value = 10 ** order_of_magnitude                                                                
        return rounded_value                                                                                    # Возвращаем ближайшее число с одним значащим разрядом

for index, row in statement_format_df[statement_format_df['symbol_contract_size'].isna()].iterrows():
    calculated_value = row['volume_lots'] * row['open_price'] / row['leverage_x'] * row['margin']               # Обновляем только строки с NaN значением в 'symbol_contract_size'
    statement_format_df.at[index, 'symbol_contract_size'] = round_down_to_significant_digit(calculated_value)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

new_column_order = ['symbol', 'position_id', 'command', 're_open', 'comm','swap',
                    'point_profit', 'symbol_contract_size', 'volume_lots', 'rate_profit', 'calc_profit_2', 'calc_crm_profit_check_2', 'profit_crm', 'spread_x', 'profit',
                    'lead_id',
                    'account_id', 'order_id', 'leverage_x', "margin", 'profit_right',  'point_profit_2', 
                    #'calc_profit', 'calc_crm_profit_check',
                    'current_rate', 'open_price', 'close_price', 'open_time', 'close_time','mapping', #'sell_last_value', 'buy_last_value', 
                    '_symbol','symbol_description', 'symbol_profit', 'symbol_margin', 'symbol_digits','symbol_point', 'symbol_calc_mode',
                    'Time', 'CloseTime', 'TimeMsc', 'CloseTimeMsc','type']
statement_format_df = statement_format_df.reindex(columns=new_column_order)

Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\statement_format_df.csv


C:\Users\nigilist\AppData\Local\Temp\ipykernel_23168\2760757265.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  statement_format_df["rate_profit"].fillna(1, inplace=True)              # заполняет все NaN значения в колонке rate_profit значением 1 и делает это "на месте,"


 
 Функция добавления новых колонок с пустыми значениями и изменения положения этих колонок  
 list_col_name = ['old_close_price'], new_position = 1
Перемещена колонка 'old_close_price' на позицию 1
 
 Функция добавления новых колонок с пустыми значениями и изменения положения этих колонок  
 list_col_name = ['close_price'], new_position = 1
Перемещена колонка 'close_price' на позицию 1


C:\Users\nigilist\AppData\Local\Temp\ipykernel_23168\2760757265.py:71: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  errors_close_price_closed_deals_df = pd.concat([errors_close_price_closed_deals_df, statement_format_df.loc[[index]]], ignore_index=True)



 Закрытые сделки с ОТСУТСТВУЮЩЕЙ ценой закрытия [errors_close_price_closed_deals_df] 
 Цена закрытия заменена расчётной, указана старая цена ['old_close_price']


,account_id,close_price,old_close_price,lead_id,order_id,symbol,volume_lots,command,re_open,position_id,swap,spread_x,open_price,current_rate,profit_right,point_profit,open_time,close_time,rate_profit,mapping,leverage_x,margin,sell_last_value,buy_last_value,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,profit,profit_crm,symbol_calc_mode,symbol_contract_size,Time,CloseTime,TimeMsc,CloseTimeMsc,type,comm
0,215034,0.66218,0.0,215036,38481,AUD/USD,1.0,1,0,3,NaN,39.0,0.66190,0.66218,66190.0000,0.66190,2023-12-04 18:14:14,2023-12-04 18:20:07,-0.000423,AUDUSD,400,166.13,0.62849,0.62856,AUDUSD,Australian Dollar vs US Dollar,USD,AUD,5,0.01,11.00,-28.00,0,100000.0,1701706454,1701706807,1701706454198,1701706807457,trade,-1
1,219838,0.00000,0.0,219840,91096,DOGE/USD,1361.0,0,0,3,NaN,0.0,0.14769,0.00000,-2010.0609,-0.14769,2024-04-27 21:28:24,2024-11-18 18:20:34,1.000000,DOGUSD,20,100.61,0.32440,0.32580,DOGEUSD,DOGE vs. USD,USD,USD,2,1.00,-2010.06,-2010.06,2,10.0,1714242504,1731946834,1714242504356,1731946834570,trade,1
2,248824,21.01000,0.0,250064,188509,Bitwise LTD,0.3,1,0,3,NaN,-30.0,21.00930,21.00930,63027.9000,21.00930,2024-12-09 22:36:03,2024-12-09 22:36:14,0.000000,Bitwise_LTD,100,630.32,5.83300,5.83300,DOTUSD,DOT vs. USD,USD,USD,2,0.01,-30.00,0.00,2,10000.0,1733776563,1733776574,1733776563568,1733776574785,trade,-1
3,248824,20.46000,0.0,250064,188725,Bitwise LTD,0.3,1,0,3,NaN,-31.0,20.45931,20.45931,61377.9300,20.45931,2024-12-10 12:55:38,2024-12-10 12:55:54,0.000000,Bitwise_LTD,100,613.78,5.83300,5.83300,DOTUSD,DOT vs. USD,USD,USD,2,0.01,-31.00,0.00,2,10000.0,1733828138,1733828154,1733828138387,1733828154697,trade,-1
4,248824,20.41000,0.0,250064,188747,Bitwise LTD,0.3,1,0,3,NaN,-28.1,20.40731,20.40731,61221.9300,20.40731,2024-12-10 13:16:00,2024-12-10 13:17:01,0.000000,Bitwise_LTD,100,612.22,5.83300,5.83300,DOTUSD,DOT vs. USD,USD,USD,2,0.01,-28.10,0.00,2,10000.0,1733829360,1733829421,1733829360302,1733829421227,trade,-1
5,248824,22.61000,0.0,250064,189810,Bitwise LTD,0.3,0,0,3,NaN,-20.0,22.60759,22.60759,-67822.7700,-22.60759,2024-12-11 18:00:32,2024-12-11 18:00:41,-0.000000,Bitwise_LTD,100,677.88,5.83300,5.83300,DOTUSD,DOT vs. USD,USD,USD,2,0.01,-20.00,0.00,2,10000.0,1733932832,1733932841,1733932832688,1733932841340,trade,1
6,248824,8.77000,0.0,250064,192023,Bitwise LTD,0.1,0,0,3,NaN,-40.7,8.76740,8.76740,-8767.4000,-8.76740,2024-12-17 11:18:03,2024-12-17 11:20:47,-0.000000,Bitwise_LTD,100,87.67,5.83300,5.83300,DOTUSD,DOT vs. USD,USD,USD,2,0.01,-40.70,0.00,2,10000.0,1734427083,1734427247,1734427083318,1734427247856,trade,1


 
 Закрытые сделки с ошибкми в цене закрытия [statement_format_df]


,account_id,close_price,old_close_price,lead_id,order_id,symbol,volume_lots,command,re_open,position_id,swap,spread_x,open_price,current_rate,profit_right,point_profit,open_time,close_time,rate_profit,mapping,leverage_x,margin,sell_last_value,buy_last_value,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,profit,profit_crm,symbol_calc_mode,symbol_contract_size,Time,CloseTime,TimeMsc,CloseTimeMsc,type,comm
44295,219838,0.0,0.0,219840,91096,DOGE/USD,1361.0,0,0,3,NaN,0.0,0.14769,0.0,-2010.0609,-0.14769,2024-04-27 21:28:24,2024-11-18 18:20:34,1.0,DOGUSD,20,100.61,0.3244,0.3258,DOGEUSD,DOGE vs. USD,USD,USD,2,1.0,-2010.06,-2010.06,2,10.0,1714242504,1731946834,1714242504356,1731946834570,trade,1


In [ ]:
# [ОБЯЗАТЕЛЕН] Создание данныж по SWOP <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
# Если значение отсутствует то установить 0.00;
# Если значение есть то установить формат float с округлением до второго знака после запятой;
# Из строк в которых не удалось произвести эти действия создать ДФ;

# Замена NaN значений на 0
statement_format_df['swap'].fillna(0, inplace=True)
statement_format_df['spread_x'].fillna(0, inplace=True)

statement_format_df['deals_swap'] = round(statement_format_df['swap'] + statement_format_df['spread_x'], 2) # Сумма свопов с учётом спреда
pd_set_option("statement_format_df", statement_format_df, 3)

def process_deals_swap_column(df, column_name='deals_swap'):
    problematic_indices = set()  # Для хранения индексов строк с ошибками

    for index, value in df[column_name].items():
        print(f"index: {index}, value: {value}")
        try:
            if pd.isna(value):                                      # Если значение отсутствует, устанавливаем 0.00
                df.at[index, column_name] = 0.00
            else:                                                   # Пробуем преобразовать значение в float и округлить
                df.at[index, column_name] = round(float(value), 2)
        except Exception as e:                                      # Если преобразование не удалось, сохраняем индекс строки
            print(f"Ошибка в строке {index}, колонка '{column_name}', значение: {value}")
            print(f"Тип значения: {type(value)}")
            print(f"Сообщение об ошибке: {e}")
            problematic_indices.add(index)

    problematic_df = df.loc[list(problematic_indices)].copy()       # Создаём DataFrame из строк, где возникли ошибки
    return df, problematic_df

statement_format_df, problematic_deals_swap_df = process_deals_swap_column(statement_format_df, 'deals_swap')

# Выводим строки с ошибками (если они есть)
if not problematic_deals_swap_df.empty:
    pd_set_option("\n Найденные строки с ошибками:", problematic_swap_df, 5)
else:
    print("\n Все значения SWOP успешно обработаны.")

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>symbol</th>
      <th>position_id</th>
      <th>command</th>
      <th>re_open</th>
      <th>comm</th>
      <th>swap</th>
      <th>point_profit</th>
      <th>symbol_contract_size</th>
      <th>volume_lots</th>
      <th>rate_profit</th>
      <th>calc_profit_2</th>
      <th>calc_crm_profit_check_2</th>
      <th>profit_crm</th>
      <th>spread_x</th>
      <th>profit</th>
      <th>lead_id</th>
      <th>account_id</th>
      <th>order_id</th>
      <th>leverage_x</th>
      <th>margin</th>
      <th>profit_right</th>
      <th>point_profit_2</th>
      <th>current_rate</th>
      <th>open_price</th>
      <th>close_price</th>
      <th>open_time</th>
      <th>close_time</th>
      <th>mapping</th>
      <th>_symbol</th>
      <th>symbol_description</th>
      <th>symbol_profit</th>
      <th>symbol_margin</th>
      <th>symbol_digits</th>
      <th>symbol_point</th>
      <th>symbol_calc_mode</th>
      <th>Time</th>
      <th>CloseTime</th>
      <th>TimeMsc</th>
      <th>CloseTimeMsc</th>
      <th>type</th>
      <th>deals_swap</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>_BAYER</td>
      <td>3</td>
      <td>0</td>
      <td>0</td>
      <td>1</td>
      <td>-2.83000</td>
      <td>-0.72000</td>
      <td>100.00000</td>
      <td>1.00000</td>
      <td>0.01083</td>
      <td>-0.78000</td>
      <td>0.00000</td>
      <td>-0.78000</td>
      <td>-0.01000</td>
      <td>-0.79000</td>
      <td>163014</td>
      <td>163012</td>
      <td>260</td>
      <td>10</td>
      <td>5.60000</td>
      <td>-72.00000</td>
      <td>NaN</td>
      <td>50.68500</td>
      <td>51.40500</td>
      <td>50.68500</td>
      <td>2023-07-03 12:07:42</td>
      <td>2023-07-04 10:00:21</td>
      <td>BAYN.XE</td>
      <td>_BAYER</td>
      <td></td>
      <td>EUR</td>
      <td>EUR</td>
      <td>3</td>
      <td>0.01000</td>
      <td>2</td>
      <td>1688375262</td>
      <td>1688454021</td>
      <td>1688375262253</td>
      <td>1688454021005</td>
      <td>trade</td>
      <td>-2.84000</td>
    </tr>
    <tr>
      <th>...</th>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
    </tr>
    <tr>
      <th>133222</th>
      <td>PLATINUM</td>
      <td>1</td>
      <td>0</td>
      <td>0</td>
      <td>1</td>
      <td>0.00000</td>
      <td>-9.90000</td>
      <td>100.00000</td>
      <td>0.01000</td>
      <td>1.00000</td>
      <td>-9.90000</td>
      <td>0.00000</td>
      <td>-9.90000</td>
      <td>-2.80000</td>
      <td>-12.70000</td>
      <td>267250</td>
      <td>265987</td>
      <td>209249</td>
      <td>100</td>
      <td>9.73000</td>
      <td>-9.90000</td>
      <td>NaN</td>
      <td>963.10000</td>
      <td>973.00000</td>
      <td>NaN</td>
      <td>2025-01-24 23:56:29</td>
      <td>NaT</td>
      <td>XPTUSD</td>
      <td>PLATINUM</td>
      <td></td>
      <td>USD</td>
      <td>USD</td>
      <td>2</td>
      <td>0.01000</td>
      <td>2</td>
      <td>1737755789</td>
      <td>0</td>
      <td>1737755789491</td>
      <td>0</td>
      <td>trade</td>
      <td>-2.80000</td>
    </tr>
  </tbody>
</table>
<p>133223 rows × 41 columns</p>
</div>

In [ ]:
filtr_df = statement_format_df[(statement_format_df['position_id'] == 3) & ((~statement_format_df["close_price"].notna()) | (statement_format_df["close_price"] == 0))]
acc_not_close_price_set = set(filtr_df['account_id'])
print(acc_not_close_price_set)
pd_set_option("ОТФИЛЬТРОВАННЫЙ ДФ", filtr_df, 50)

In [ ]:
filtr_df = statement_format_df[statement_format_df["position_id"] == 115419]
pd_set_option("statement_df[""] ==", filtr_df, 3)

In [ ]:
# определяем сделки по символам SHIB/USD <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<https://fintech-area.slack.com/archives/G0165A4V6M7/p1730392923722159?thread_ts=1730248723.527459&cid=G0165A4V6M7
pd.options.display.float_format = '{:,.5f}'.format                                                                      # отображение до 5 знаков после запятой
filtered_rows = statement_format_df[(statement_format_df['symbol']== "SHIB/USD")]                               # определяем сделки по которым у символов нет размера контрактов
filtered_rows = filtered_rows.sort_values(by="calc_crm_profit_check_2", ascending=False)                                # ascending=False для сортировки по убыванию
print(f"Счета у которых есть сделки по SHIB/USD: \n", set(filtered_rows['account_id']))
#print(f"\n Счета которые нужно исключить: \n", set(filtered_rows['account_id']) - set([69806, 157572,179936, 233233]))
print(f"\n Сделки по символам без конфигурационных файлов: \n", filtered_rows[['account_id','order_id','symbol', 'volume_lots']])

In [ ]:
pd.options.display.float_format = '{:,.5f}'.format  # отображение до двух знаков после запятой

statement_format_df = statement_format_df.sort_values(by="calc_crm_profit_check_2", ascending=True)  # ascending=False для сортировки по убыванию
pd_set_option("ОТФОРМАТИРОВАННЫЕ Сделки и символы для генерации истории", statement_format_df, 5)

statement_format_df = statement_format_df.sort_values(by="calc_crm_profit_check_2", ascending=False)  # ascending=False для сортировки по убыванию
pd_set_option("ОТФОРМАТИРОВАННЫЕ Сделки и символы для генерации истории", statement_format_df, 5)

statement_format_df = statement_format_df.sort_values(by="calc_crm_profit_check_2", ascending=True)  # ascending=False для сортировки по убыванию
pd_set_option("ОТФОРМАТИРОВАННЫЕ Сделки и символы для генерации истории", statement_format_df, 5)

statement_format_df = statement_format_df.sort_values(by="calc_crm_profit_check_2", ascending=False)  # ascending=False для сортировки по убыванию
pd_set_option("ОТФОРМАТИРОВАННЫЕ Сделки и символы для генерации истории", statement_format_df, 5)

#filtered_rows = statement_format_df[(statement_format_df['mapping'] == 'NG') & (statement_format_df['close_price'] > 0)]
filtered_rows = statement_format_df[(statement_format_df['mapping'] == 'AUDUSD')]  #"""& (statement_format_df['account_id'] == 233254)"""

statement_format_df = statement_format_df.sort_values(by="calc_crm_profit_check_2", ascending=True)  # ascending=False для сортировки по убыванию
pd_set_option("ОТФОРМАТИРОВАННЫЕ Сделки и символы для генерации истории", statement_format_df, 50)

# Установка опций отображения без научной нотации
pd.options.display.float_format = '{:,.5f}'.format  # отображение до двух знаков после запятой
pd_set_option("ОТФОРМАТИРОВАННЫЕ Сделки и символы для генерации истории", statement_format_df, 50)

In [ ]:
pd_set_option("О", balance_df, 3)
pd_set_option("О", statement_format_df, 3)

In [ ]:
# Проверка корректности индексов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
common_columns = set(balance_df.columns).intersection(set(statement_format_df.columns))
print("Общие столбцы:", common_columns)

balance_df = balance_df.reset_index(drop=True)
statement_format_df = statement_format_df.reset_index(drop=True)

# Проверим соответствие типов данных для общих столбцов
for col in common_columns:
    if balance_df[col].dtype != statement_format_df[col].dtype:
        print(f"Типы данных для столбца '{col}' не совпадают:")
        print(f"Тип в df1: {balance_df[col].dtype}, Тип в df2: {statement_format_df[col].dtype}")
    else:
        print(f"Типы данных для столбца '{col}' совпадают.")

def check_index_continuity(df):
    # Проверим, является ли индекс RangeIndex, который автоматически создается от 0 до n-1
    if isinstance(df.index, pd.RangeIndex) and df.index.start == 0 and df.index.stop == len(df) and df.index.step == 1:
        print("Индекс корректен: он идет от 0 до n-1 без пропусков и дублей.")
    else:
        print("Индекс некорректен: есть пропуски, дубли, или индекс не начинается с 0.")
        # Выводим дополнительные сведения
        print("Уникален ли индекс?", df.index.is_unique)
        print("Минимальное значение индекса:", df.index.min())
        print("Максимальное значение индекса:", df.index.max())
        print("Длина индекса:", len(df.index))
        print("Ожидаемая длина:", df.index.max() + 1)
        missing_indices = set(range(len(df))) - set(df.index)
        if missing_indices:
            print("Отсутствующие значения индекса:", sorted(missing_indices))
        else:
            print("В индексе нет пропусков, но может быть нарушен порядок.")

print("Проверка корректного DataFrame:")
check_index_continuity(balance_df)
print("Проверка корректного DataFrame:")
check_index_continuity(statement_format_df)

# Получаем максимальное значение индекса в statement_format_df
n_max = statement_format_df.index.max()
# Переиндексируем balance_df, начиная с (n_max + 1)
balance_df = balance_df.set_index(pd.RangeIndex(start=n_max + 1, stop=n_max + 1 + len(balance_df)))
# Проверяем результат
print("Максимальное значение индекса в statement_format_df:", n_max)
print("Новый индекс balance_df:", balance_df.index)


In [ ]:
# Сохраняем первые три строки обоих DataFrame в разные CSV файлы
balance_df.to_csv('balance_df.csv', index=False)
statement_format_df.to_csv('statement_format_df.csv', index=False)

del balance_df, statement_format_df
# Загружаем данные из CSV файлов обратно в новые DataFrame
balance_df = pd.read_csv('balance_df.csv')
statement_format_df = pd.read_csv('statement_format_df.csv')

# Объединяем загруженные DataFrame по строкам (axis=0), игнорируя индексы
#combined_df = pd.concat([balance_df, statement_format_df], axis=0, ignore_index=True)

# Выводим результат объединения


In [ ]:
# Объединение построчно с обработкой ошибок
combined_df = balance_df.copy()  # Начинаем с df1
df2 = statement_format_df
for i in range(len(df2)):  # Проходим по строкам df2
    try:
        # Пытаемся объединить текущую строку из df2 с combined_df
        combined_df = pd.concat([combined_df, df2.iloc[[i]]], ignore_index=True)
    except Exception as e:
        print(f"Ошибка при добавлении строки {i}: {df2.iloc[i].to_dict()}")
        print(f"Ошибка: {e}")


#combined_df = pd.concat([balance_df, statement_format_df],  axis=0, ignore_index=True)

In [16]:
# [ОБЯЗАТЕЛЕН] Объединение DF [БАЛАНСОВЫХ] операций и DF [СДЕЛОК] <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#```````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
statement_format_account_id_set = set(statement_format_df["account_id"])
balance_account_id_set = set(balance_df["account_id"])

if statement_format_account_id_set == balance_account_id_set:
    print("Множества ТОЖДЕСТВЕННЫ счетов ДФ балансовых операций и ДФ торговых операций")
else:
    print("Множества различны")
    print("Элементы только в set1:", statement_format_account_id_set - balance_account_id_set)
    print("Элементы только в set2:", balance_account_id_set - statement_format_account_id_set)

combined_df = pd.concat([statement_format_df, balance_df],  axis=0, ignore_index=True)          # Объединение двух DataFrame
combined_df_sorted = combined_df.sort_values(by='TimeMsc', ascending=True)                      # Сортировка по колонке TimeMsc по возрастанию

def columns_to_int64(columns_list):                                                             # Форматирование Данных столбцов в [int]
    for column in columns_list:
        combined_df_sorted[column] = pd.to_numeric(combined_df_sorted[column], errors='coerce').astype('Int64')
columns_to_int64(['id', 'account_id', 'TimeMsc', 'Time', 'account_id', 'command', 'CloseTime', 'CloseTimeMsc'])        

# Объединение колонок ['id'таблица балансовых операций] и ['order_id'таблица торговых операций] <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
conflict_df = combined_df_sorted[(combined_df_sorted['id'].notna()) & (combined_df_sorted['order_id'].notna())] # Найдём строки, где оба значения 'id' и 'order_id' присутствуют
if not conflict_df.empty: pd_set_option("Найдены строки, где значения присутствуют одновременно в колонках 'id' и 'order_id':", onflict_df, 3)
else: print("Нет строк, где значения присутствуют одновременно в обеих колонках 'id' и 'order_id'.")
combined_df_sorted['crm_deals_id'] = combined_df_sorted['id'].combine_first(combined_df_sorted['order_id'])     # 'crm_deals_id' с объединением значений из 'id' и 'order_id'
combined_df_sorted = combined_df_sorted.drop(columns=['id', 'order_id'])                                        # Удалим колонки 'id' и 'order_id'

pd_set_option("Таблица ТОРГОВЫХ и БАЛАНСОВЫХ операций",combined_df_sorted, 3)                  

Множества различны
Элементы только в set1: set()
Элементы только в set2: {241025, 260355, 70396, 242185, 266878, 265485, 264078, 265742, 267024, 234510, 267539, 241685, 203413, 239771, 267772, 260258, 266275, 219426, 107559, 243752, 80295, 103983, 239666, 268084, 260279, 234041, 62265, 235580, 242109, 267965, 239552, 238145, 265922, 268227, 265030, 221255, 237515, 241740, 143820, 261710, 243025, 267734, 264411, 267356, 259547, 256222, 234462, 58337, 268002, 26596, 233705, 237290, 264684, 234479, 258673, 238450, 138738, 239865, 73852, 237694, 86783}


Нет строк, где значения присутствуют одновременно в обеих колонках 'id' и 'order_id'.
Таблица ТОРГОВЫХ и БАЛАНСОВЫХ операций


,symbol,position_id,command,re_open,comm,swap,point_profit,symbol_contract_size,volume_lots,rate_profit,calc_profit_2,calc_crm_profit_check_2,profit_crm,spread_x,profit,lead_id,account_id,leverage_x,margin,profit_right,point_profit_2,current_rate,open_price,close_price,open_time,close_time,mapping,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_calc_mode,Time,CloseTime,TimeMsc,CloseTimeMsc,type,deals_swap,balance_transactions,comment,finance_type,crm_deals_id
0,_BAYER,3.00000,0,0.00000,1.00000,-2.83000,-0.72000,100.00000,1.00000,0.01083,-0.78000,0.00000,-0.78000,-0.01000,-0.79000,"163,014.00000",163012,10.00000,5.60000,-72.00000,NaN,50.68500,51.40500,50.68500,2023-07-03 12:07:42,2023-07-04 10:00:21,BAYN.XE,_BAYER,,EUR,EUR,3.00000,0.01000,2.00000,1688375262,1688454021,1688375262253,1688454021005,trade,-2.84000,NaN,NaN,NaN,260.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141660,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,239742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1738014455,<NA>,1738014455000,<NA>,balance,NaN,567.00000,referrals,correction,233.00000


In [ ]:
# Шаг 1: Найдем минимальное значение в колонке Time
min_time = combined_df_sorted[combined_df_sorted['position_id'] == 1]['Time'].min()

# Шаг 2: Проверим, сколько раз оно встречается
count_min_time = (combined_df_sorted['Time'] == min_time).sum()

# Шаг 3: Если минимальное значение встречается более одного раза, выводим соответствующие строки
if count_min_time > 1:
    print(combined_df_sorted[combined_df_sorted['Time'] == min_time])
else:
    print(f"Минимальное значение [{min_time}] встречается только один раз.")

# Фильтрация строк, где значение в колонке 'CloseTime' меньше 1602524033
filtered_df = combined_df_sorted[(combined_df_sorted['CloseTime'] < min_time) & (combined_df_sorted['position_id'] == 3)]

# Вывод отфильтрованного DataFrame
pd_set_option("Таблица Сделок с временем закрытия меньшей чем время открытия",filtered_df, 3)

In [ ]:
# сортировка счетов по возрасту последней закрытой сделки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````
result_df = pd.DataFrame(columns=['account_id', 'CloseTime', 'close_time'])                 # Создадим пустой DataFrame для результатов

for account_id, group in combined_df_sorted.groupby('account_id'):                          # Группируем по уникальным `account_id`
    sorted_group = group.sort_values(by='Time', ascending=False)                            # Сортируем группу по убыванию 'Time'
    close_time_row = sorted_group[sorted_group['CloseTime'] > 0]                            # Фильтруем строки, где `CloseTime > 0`
    
    if not close_time_row.empty:                                                            # Если найдено значение `CloseTime > 0`, добавляем его в результат
        max_close_time_row = close_time_row.iloc[0]                                         # Берем первую строку после сортировки
        result_df = pd.concat([result_df, pd.DataFrame({
            'account_id': [account_id],
            'CloseTime':  [max_close_time_row['CloseTime']],
            'close_time': [max_close_time_row['close_time']]

        })], ignore_index=True)
    else:
        result_df = pd.concat([result_df, pd.DataFrame({                                    # Если не найдено `CloseTime > 0`, добавляем None в значения `CloseTime`
            'account_id': [account_id],
            'CloseTime': [None]
        })], ignore_index=True)

result_df = result_df.sort_values(by='CloseTime', ascending=True).reset_index(drop=True)
pd_set_option("Таблица",result_df, 30)

In [ ]:
# Сохраняем базу; Формируем множество торговых счетов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
file_path = 'combined_df_sorted.csv'                                            # Сохранение DataFrame combined_df_sorted в CSV файл
combined_df_sorted.to_csv(file_path, index=False, sep=',', encoding='utf-8')
print(f"DataFrame сохранён в файл {file_path}")

"""unique_values = set(combined_df_sorted['account_id'])                           # Создание множества из колонки
acc_set = list({int(x) for x in unique_values})                                 # Преобразуем значения множества в целые числа
print("acc_set = ", acc_set)"""


In [ ]:
acc_set = list({int(x) for x in unique_values})                                 # Преобразуем значения множества в целые числа
open('files/acc_list.csv', 'w').close()                                         # Создаем пустой файл # Открываем файл в режиме записи и сразу закрываем
with open('files/acc_list.csv', 'w') as f:                                      # Открываем файл для записи
    f.write(', '.join(map(str, acc_set)))                                       # Преобразуем элементы множества в строки и записываем их, разделяя запятыми

In [ ]:
# Фильтрация счетов с исключительно закрытыми сделками <<
query_accounts_without_positions = """
SELECT GROUP_CONCAT(DISTINCT account_id) AS account_ids_list
FROM `br-stone`.trades
WHERE position_id = 3;"""

query_accounts_without_positions_crm_df = pd_read_sql(query_accounts_without_positions)

# Извлечение строки, разделение по запятым и преобразование в список целых чисел
account_ids = query_accounts_without_positions_crm_df.iloc[0, 0]                                # Извлекаем строку из первой ячейки
account_ids_list = list(set([int(id) for id in account_ids.split(',')]))                        # Разделяем и преобразуем

print("Количество торговых счетов без открытых сделок:", len(account_ids_list))

filtered_df = combined_df_sorted[~combined_df_sorted['account_id'].isin(account_ids_list)]

pd_set_option("без открытых сделок",filtered_df, 3)


In [ ]:
# Проверка на дублирующиеся названия колонок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
duplicate_columns = combined_df_sorted.columns[combined_df_sorted.columns.duplicated(keep=False)]# keep=False для отображения всех дубликатов
if duplicate_columns.any():
    print("ERROR: Дубликаты в названиях колонок:")
    print(duplicate_columns.unique())  # Выводим только уникальные дублирующиеся названия
else:
    print("[combined_df_sorted]: Дубликатов в названиях колонок нет.")

combined_df_sorted = combined_df_sorted.reset_index(drop=True)
print("[combined_df_sorted] индексы сброшены")


In [9]:
combined_df_sorted = pd.read_csv("files/migration_errors/migration_errors_250128/Таблица НеСозданных объектов OPEN сделки.csv", encoding='utf-8') # Загружаем данные из CSV файла
combined_df_sorted["mt5error"] = None
pd_set_option("Таблица ТОРГОВЫХ и БАЛАНСОВЫХ операций",combined_df_sorted, 10)    

Таблица ТОРГОВЫХ и БАЛАНСОВЫХ операций


,symbol,position_id,command,re_open,comm,swap,point_profit,symbol_contract_size,volume_lots,rate_profit,calc_profit_2,calc_crm_profit_check_2,profit_crm,spread_x,profit,lead_id,account_id,leverage_x,margin,profit_right,point_profit_2,current_rate,open_price,close_price,open_time,close_time,mapping,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_calc_mode,Time,CloseTime,TimeMsc,CloseTimeMsc,type,deals_swap,balance_transactions,comment,finance_type,crm_deals_id,open_deals_id,close_deals_id,positions_id,mt5error,deal_profit,deal_profit_dif,deal_profit_dif_per
0,AUD/USD,3,1,0,-1,0.0,0.66190,100000,1.00,-0.000423,-27.999694,3.062000e-04,-28.00,39.0,11.00,215036,215034,400.0,166.13,66190.0000,NaN,0.66218,0.66190,0.66218,2023-12-04 18:14:14,2023-12-04 18:20:07,AUDUSD,AUDUSD,Australian Dollar vs US Dollar,USD,AUD,5,0,0.0,1701706454,1701706807,1701706454198,1701706807457,trade,39.0,NaN,NaN,NaN,38481,-1,-1,-1,None,NaN,NaN,NaN
1,_E.ON,3,0,0,1,0.0,-0.13500,100,0.20,-106.429630,287.360000,9.995347e-10,287.36,-300.0,-12.64,194865,194863,1.0,270.23,-2.7000,NaN,12.38900,12.52400,12.38900,2023-12-12 15:17:41,2024-05-27 10:05:39,EOAN.XE,_E.ON,NaN,EUR,EUR,2,0,2.0,1702387061,1716793539,1702387061372,1716793539834,trade,-300.0,NaN,NaN,NaN,41253,-1,-1,-1,None,NaN,NaN,NaN
2,Deutsche Lufthansa AG,3,0,0,1,0.0,-2.41500,100,0.03,-27.692202,200.630000,1.240002e-08,200.63,-250.0,-49.37,207595,207593,1.0,26.89,-7.2450,NaN,5.73600,8.15100,5.73600,2023-12-15 18:06:03,2024-07-23 10:08:21,LHAG.DE,LHA,NaN,EUR,EUR,2,0,2.0,1702656363,1721718501,1702656363270,1721718501403,trade,-250.0,NaN,NaN,NaN,42762,-1,-1,-1,None,NaN,NaN,NaN
3,AUD/JPY,3,0,0,1,0.0,-1.09000,100000,0.03,-0.000850,2.779991,-9.500000e-06,2.78,-1.0,1.78,234665,234672,400.0,5.13,-3270.0000,NaN,96.46300,97.55300,96.46300,2023-12-27 16:56:45,2023-12-28 10:08:18,AUDJPY,AUDJPY,Australian Dollar vs Japanese Yen,JPY,AUD,3,0,0.0,1703689005,1703750898,1703689005121,1703750898322,trade,-1.0,NaN,NaN,NaN,46216,-1,-1,-1,None,NaN,NaN,NaN
4,AUD/JPY,3,1,0,-1,0.0,-7.29800,100000,0.01,-0.021036,153.519998,-1.800000e-06,153.52,-300.0,-146.48,235404,235415,400.0,1.71,-7298.0000,NaN,103.88700,96.58900,103.88700,2023-12-28 18:33:26,2024-06-14 13:37:08,AUDJPY,AUDJPY,Australian Dollar vs Japanese Yen,JPY,AUD,3,0,0.0,1703781206,1718361428,1703781206994,1718361428465,trade,-300.0,NaN,NaN,NaN,46975,-1,-1,-1,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10,EUR/HUF,3,1,0,-1,0.0,-8.42000,100000,0.01,-0.000847,7.129972,-2.820000e-05,7.13,-100.0,-92.87,243168,241971,400.0,2.77,-8420.0000,NaN,400.93000,392.51000,400.93000,2024-09-05 11:06:23,2024-10-15 10:21:31,EURHUF,EURHUF,Euro vs Hungarian Forint,HUF,EUR,3,0,0.0,1725523583,1728976891,1725523583735,1728976891324,trade,-100.0,NaN,NaN,NaN,141376,-1,-1,-1,None,NaN,NaN,NaN
11,GBP/CAD,3,1,0,-1,0.0,-0.00924,100000,0.02,-0.362013,6.690000,5.519994e-08,6.69,-50.0,-43.31,97499,97498,400.0,6.49,-18.4800,NaN,1.80189,1.79265,1.80189,2024-10-16 11:39:32,2024-10-28 11:00:56,GBPCAD,GBPCAD,Great Britain Pound vs Canadian Dollar,CAD,GBP,5,0,0.0,1729067972,1730106056,1729067972251,1730106056996,trade,-50.0,NaN,NaN,NaN,161642,-1,-1,-1,None,NaN,NaN,NaN
12,K+S AG,3,0,0,1,0.0,-0.04000,100,0.05,-248.950000,49.790000,-1.087130e-12,49.79,-350.0,-300.21,247423,246204,1.0,60.60,-0.2000,NaN,11.40000,11.44000,11.40000,2024-11-19 10:10:43,2024-12-16 11:00:40,SDFGn.DE,SDF,NaN,EUR,EUR,2,0,2.0,1732003843,1734339640,1732003843915,1734339640968,trade,-350.0,NaN,NaN,NaN,177546,-1,-1,-1,None,NaN,NaN,NaN
13,AUD/USD,3,0,0,1,0.0,-0.00087,100000,4.00,-2.017241,702.000000,2.400302e-07,702.00,400.0,1102.00,264179,262908,400.0,623.06,-348.0000,NaN,0.61925,0.62012,0.61925,2025-01-06 15:28:43,2025-01-09 11:51:43,AUDUSD,AUDUSD,Australian Dollar vs US Dollar,USD,AUD,5,0,0.0,1736170123,1736416303,1736170123704,1

In [10]:
# [ОБЯЗАТЕЛЕН] Сохраняем базу сделок в CSV, создаём ДФ сделок для миграции, создаём ДФ для ошибок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
def create_error_df(combined_df_sorted):

    unique_values = set(combined_df_sorted['account_id'])                                   # Создание множества из колонки
    acc_set = list({int(x) for x in unique_values})                                         # Преобразуем значения множества в целые числа
    
    open('files/acc_list_250125.csv', 'w').close()                                                 # Создаем пустой файл # Открываем файл в режиме записи и сразу закрываем
    with open('files/acc_list__250125.csv', 'w') as f: f.write(', '.join(map(str, acc_set)))        # Открываем файл для записи; элементы множества в строки, записываем их, разделяя запятыми"""

    imported["int_to_str_csv"](acc_set,"files/temp", "acc_set.csv")

    combined_df_sorted.to_csv("combined_df_sorted_250125.csv", index=False, encoding='utf-8')      # Сохраняем данные сделок для переноса в CSV
    total_deal_df = pd.read_csv("combined_df_sorted_250125.csv", encoding='utf-8')                 # Обратно загружаем данные из CSV

    total_deal_df['open_deals_id']       = None                                             # колонка для записи мт5 id после успешного совершения сделки [IN]
    total_deal_df['close_deals_id']      = None                                             # колонка для записи мт5 id после успешного совершения сделки [OUT]
    total_deal_df['positions_id']        = None                                             # колонка для записи мт5 id после успешного создания Позиции
    #total_deal_df = combined_df_sorted.copy()# Создание нового ДФ с данными по сделкам
    #del combined_df_sorted# Удаляем ДФ
    total_deal_df['mt5error']            = pd.Series(pd.NA, dtype='object')
    total_deal_df['deal_profit']         = pd.Series(np.nan, dtype='float64')
    total_deal_df['deal_profit_dif']     = pd.Series(np.nan, dtype='float64')
    total_deal_df['deal_profit_dif_per'] = pd.Series(np.nan, dtype='float64')
    columns_error_df                     = total_deal_df.columns.tolist()                   # Преобразуем колонки DataFrame в список

    total_deal_df = total_deal_df.sort_values(by="TimeMsc", ascending=True)                 # Сортировка по столбцу "TimeMsc" по возрастанию

    def create_error_df_with_mt5error(columns):                                             # def создания ДФ с переданным списком колонок
        df = pd.DataFrame(columns=columns)
        df['mt5error'] = df['mt5error'].astype('object')
        return df

    error_operatio_balance_generis_df   = create_error_df_with_mt5error(columns_error_df)
    error_deal_balance_df               = create_error_df_with_mt5error(columns_error_df)
    error_update_balance_df             = create_error_df_with_mt5error(columns_error_df)
    error_symbol_df                     = create_error_df_with_mt5error(columns_error_df)
    error_deal_open_df                  = create_error_df_with_mt5error(columns_error_df)
    error_swop_df                       = create_error_df_with_mt5error(columns_error_df)
    error_deal_close_df                 = create_error_df_with_mt5error(columns_error_df)
    error_deal_profit_df                = create_error_df_with_mt5error(columns_error_df)

    return (total_deal_df, error_operatio_balance_generis_df, error_deal_balance_df, error_update_balance_df, error_symbol_df,
            error_deal_open_df, error_swop_df, error_deal_close_df, error_deal_profit_df, acc_set)

total_deal_df, error_operatio_balance_generis_df, error_deal_balance_df, error_update_balance_df, error_symbol_df, error_deal_open_df, error_swop_df, \
error_deal_close_df, error_deal_profit_df, acc_set = create_error_df(combined_df_sorted)             # Присвоение возвращаемых DataFrame переменным

print("Торговых счетов к миграции: len(acc_set) = [", len(acc_set), "]")                             # Выводим количество счетов в работе
print("СДЕЛОК к миграции: len(total_deal_df) = [", len(total_deal_df), "]")                          # Выводим количество счетов в работе

total_deal_df = total_deal_df.sort_values(by='TimeMsc', ascending=True).reset_index(drop=True)

Список из [15] элементов, сохранён в файл: files/temp\2025-01-29 16-31-25.915acc_set.csv
Торговых счетов к миграции: len(acc_set) = [ 15 ]
СДЕЛОК к миграции: len(total_deal_df) = [ 15 ]


In [ ]:
del error_operatio_balance_generis_df, error_deal_balance_df, error_update_balance_df, error_symbol_df, error_deal_open_df, error_swop_df, error_deal_close_df, error_deal_profit_df

In [ ]:
# Считаем количество вхождений для каждого уникального значения в 'account_id' и сортируем по убыванию <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
noun = 10
pd.set_option('display.max_rows', 1000)
account_id_counts = total_deal_df['account_id'].value_counts().sort_values(ascending=False)
filtered_account_id_counts = account_id_counts[account_id_counts < noun]
print(len(filtered_account_id_counts), filtered_account_id_counts)

In [19]:
#1
account_ids = [266387, 266205, 266172, 266162, 266137, 266059, 266031, 266016, 265977, 265949, 265941, 265919, 265915, 265909, 265873, 265833, 265831, 265824, 265721, 265717, 265685, 265680, 265673, 265633, 265591, 265581, 265579, 265558, 265488, 265463, 265455, 265453, 265451, 265441, 265395, 265389, 265357, 265344, 265338, 265328, 265318, 265315, 265298, 265294, 265285, 265276, 265274, 265207, 265168, 265152]
acc_set = [x for x in acc_set if x not in account_ids]


"""#2
acc_set = [266387, 266370, 266273, 266265, 266213, 266205, 266172, 266137, 
 266059, 266031, 266016, 265987, 265977, 265949, 265941, 265932, 
 265919, 265915, 265909, 265873, 265833, 265831, 265824, 265721, 
 265717, 265685, 265673, 265633, 265591, 265581, 265558, 265488, 
 265463, 265455, 265453, 265451, 265441, 265395, 265389, 265357, 
 265344, 265338, 265328, 265318, 265315, 265298, 265294, 265285, 
 265276, 265274]# 2"""

acc_set = [69278 , 237660 , 217701]


In [ ]:
print("len(acc_set) = [", len(acc_set), "]; acc_set:")                                                  # Выводим количество счетов в работе
print(acc_set)

open('files/acc_list.csv', 'w').close()                                                 # Создаем пустой файл # Открываем файл в режиме записи и сразу закрываем
with open('files/acc_list.csv', 'w') as f: f.write(', '.join(map(str, acc_set)))        # Открываем файл для записи; элементы множества в строки, записываем их, разделяя запятыми

In [ ]:
# Фильтрация сделок по одному клиенту / клиентам <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#```````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
#total_deal_df = total_deal_df[total_deal_df['account_id'].isin(acc_set)].copy()

colums_list = ['crm_deals_id', 'symbol_digits', 'open_deals_id', 'close_deals_id', 'positions_id', 
                  'account_id', 'TimeMsc','Time', 'account_id', 'command','CloseTime', 'CloseTimeMsc', 'position_id', 're_open', 'comm', 'symbol_contract_size']

def columns_to_int64(columns_list):
    for column in columns_list:
        total_deal_df[column] = pd.to_numeric(total_deal_df[column], errors='coerce').astype('Int64')
columns_to_int64(colums_list) 

total_deal_df = total_deal_df.sort_values(by='TimeMsc', ascending=True)
total_deal_df = total_deal_df.reset_index(drop=True)
"""
rows = total_deal_df.iloc[5:7]  # Возвращает строки с индексами 1 и 2
display(rows)"""

pd_set_option("total_deal_df ДФ готовый для миграции",total_deal_df, 16)

# Сохраняем DataFrame в CSV файл <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
text = ", ".join(map(str, acc_set ))
csv_file_path = "files/statement_" #+ text+".csv"
total_deal_df.to_csv(csv_file_path, index=False)
full_path = os.path.abspath(csv_file_path)                  # 
print(f"Полный путь к сохраняемому файлу: {full_path}")

In [ ]:
filtr_df = total_deal_df[total_deal_df["crm_deals_id"] == 115419]
pd_set_option("ЗАПРОС ПОКОНКРЕТНОМУ ОРДЕРУ", filtr_df, 3)

In [ ]:
filtr_df = total_deal_df[total_deal_df['account_id'] == 254860]
pd_set_option("ЗАПРОС ПОКОНКРЕТНОМУ КЛИЕНТУ", filtr_df, 3)
total_deal_df = filtr_df

In [ ]:
# Фильтрация сделок по одному ордеру <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#```````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
order_total_deal_df = total_deal_df[total_deal_df['crm_deals_id'] == 50759].copy()
def columns_to_int64(columns_list):
    for column in columns_list:
        order_total_deal_df[column] = pd.to_numeric(order_total_deal_df[column], errors='coerce').astype('Int64')
columns_to_int64(['crm_deals_id', 'symbol_digits', 'open_deals_id', 'close_deals_id', 'positions_id', 'account_id', 'TimeMsc', 'Time',
                  'account_id', 'command','CloseTime','CloseTimeMsc']) 
order_total_deal_df = order_total_deal_df.reset_index(drop=True)
pd_set_option("[order_total_deal_df] ДФ готовый для миграции",order_total_deal_df, 3)
total_deal_df = order_total_deal_df

In [20]:
total_deal_df.sort_values(by='Time', inplace=True)

In [ ]:
# [ОБЯЗАТЕЛЕН] 
def convert_columns_to_int(df, columns, default_value):
    """
    Преобразует значения в указанных столбцах DataFrame к типу int.
    Если значение отсутствует (NaN) или не может быть преобразовано, присваивает значение по умолчанию.
    
    :param df: pandas.DataFrame
    :param columns: список столбцов для преобразования
    :param default_value: значение по умолчанию для отсутствующих или некорректных значений
    :return: pandas.DataFrame
    """
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')       # Преобразование к числу (замена некорректных значений на NaN)
        df[col] = df[col].fillna(default_value).astype(int)     # Замена NaN и преобразование к int

        print("\n", df[col].value_counts(dropna=False), "\n")

    return df

# Пример использования
pd.set_option('display.max_rows', None)
df = total_deal_df
default_value = -1
columns = [
    "position_id", "command", "re_open", "comm", "symbol_contract_size", 
    "lead_id", "account_id", "symbol_digits", "symbol_point", "Time", 
    "CloseTime", "TimeMsc", "CloseTimeMsc", "crm_deals_id", 
    "open_deals_id", "close_deals_id", "positions_id"
]
df = convert_columns_to_int(df, columns, default_value)
#display(df)

pd.set_option('display.max_rows', None)
# print("\n", total_deal_df["symbol_digits"].value_counts(dropna=False), "\n")
# print("\n", total_deal_df["command"].value_counts(dropna=False), "\n")

In [ ]:
def convert_columns_to_int_report(df, columns, default_value):
    """
    Преобразует значения в указанных столбцах DataFrame к типу int.
    Если значение отсутствует (NaN) или не может быть преобразовано, присваивает значение по умолчанию.
    Возвращает отчёт о типах данных в виде DataFrame.

    Параметры:
    df (pd.DataFrame): Исходный DataFrame.
    columns (list): Список столбцов для преобразования.
    default_value (int): Значение по умолчанию для отсутствующих или некорректных значений.

    Возвращает:
    pd.DataFrame: Отчёт о типах данных в столбцах.
    """
    report = {}
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')  # Преобразование к числу (замена некорректных значений на NaN)
        df[col] = df[col].fillna(default_value).astype(int)  # Замена NaN и преобразование к int
        value_counts = df[col].map(type).value_counts()  # Считаем типы данных в столбце
        report[col] = value_counts

    # Преобразуем отчёт в DataFrame
    report_df = pd.DataFrame(report).fillna(0).astype(int)  # Заполняем пропуски нулями и преобразуем к int
    return report_df

# Пример использования
pd.set_option('display.max_rows', None)
df = total_deal_df.copy()
default_value = -1
columns = [
    "position_id", "command", "re_open", "comm", "symbol_contract_size", 
    "lead_id", "account_id", "symbol_digits", "symbol_point", "Time", 
    "CloseTime", "TimeMsc", "CloseTimeMsc", "crm_deals_id", 
    "open_deals_id", "close_deals_id", "positions_id"
]
df = convert_columns_to_int_report(df, columns, default_value)
pd_set_option("Т",df, 3)  
pd.set_option('display.max_rows', None)

|
# Фильтр по номеру счёта

In [ ]:
# фильтр по номеру счёта <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
filtr_df = total_deal_df[total_deal_df['account_id'] == 265144]
pd_set_option("ЗАПРОС ПОКОНКРЕТНОМУ КЛИЕНТУ", filtr_df, 3)
total_deal_df = filtr_df

In [ ]:
# Чтение файла и преобразование содержимого в список int
file_path =  r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250113\ID_dials_7_way.txt"
if os.path.exists(file_path):print("Файл существует!")
else:print("Файл не найден!")
with open(file_path, 'r') as file:
    # Чтение всех строк, удаление лишних символов и преобразование в int
    deal_list_id = [int(line.strip()) for line in file]
deal_list_id.sort()
print(f"[{len(deal_list_id)}]ID сделок ко второму проходу.txt: {deal_list_id}")

In [ ]:
# Фильтрация [балансовых] сделок по списку значений [crm_deals_id] <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
crm_deals_id_list = deal_list_id                                       # Замените на ваш список значений
filtr_df = total_deal_df[(total_deal_df['crm_deals_id'].isin(crm_deals_id_list)) & (total_deal_df['type'] == 'trade')]
pd_set_option("ЗАПРОС ПО списку ID балансовых операий КЛИЕНТУ", filtr_df, 10)                    # Вывод отфильтрованного DataFrame
total_deal_df = filtr_df                                                        # Обновление total_deal_df

In [ ]:
if len(total_deal_df) >= 20:
    pd.set_option('display.max_rows', 25)
    display(total_deal_df.head(25))
else:
    print("DataFrame содержит меньше 20 строк.")

total_deal_df = total_deal_df.iloc[25:].reset_index(drop=True)
display(total_deal_df.head(25))

In [12]:
# [ОБЯЗАТЕЛЕН] 
total_deal_df = total_deal_df.reset_index(drop=True)

In [13]:
from MT5Manager import ManagerAPI, AdminAPI
import MT5Manager
manager = ManagerAPI()
admin = AdminAPI()

server_mt5_ip_port = "144.76.64.243:443"
manager_mt5_login = 1100
manager_mt5_password = "Jw-pW4Kh"

In [14]:
acc_list = acc_set                  # список счетов для нулевого пополнения 
#acc_list = [46726, ]
len(acc_list)

15

In [22]:
# Нулевое пополнение для корректного проведения балансовых операций <<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````
print(acc_list)
summ0 = 100000000                   # сумма нулевого пополнения
date0 = 1685577600                  # дата нулевого пополнения
balance_0(acc_list, summ0, date0)   # Функция из файла [imports.py]

[262146, 262151, 73740, 229393, 237585, 237591, 237602, 163878, 229419, 237613, 221255, 262218, 237646, 237656, 237657, 237660, 114780, 237678, 65649, 237689, 262266, 73852, 237694, 90243, 262291, 262298, 262302, 237728, 262308, 262317, 262320, 237747, 237748, 262324, 237757, 262335, 237764, 262340, 180429, 237776, 237778, 237779, 237796, 237797, 262391, 262399, 262411, 237841, 262422, 262424, 196889, 237854, 147748, 237870, 237871, 237878, 237895, 237904, 262480, 262483, 262490, 82283, 180587, 237940, 262523, 262524, 57730, 74114, 262540, 262552, 164260, 262570, 262572, 238002, 238004, 246204, 238013, 254397, 262593, 254406, 262604, 262606, 254419, 238048, 238056, 238068, 205301, 238070, 262647, 238078, 262655, 254464, 238081, 238097, 262681, 115229, 139807, 115232, 238111, 238114, 41508, 254500, 238119, 238122, 254508, 221741, 238136, 238138, 262720, 238145, 238155, 238163, 238165, 238170, 238172, 238177, 254561, 238180, 238181, 238185, 238194, 238197, 262776, 238205, 238210, 238214,

In [ ]:
pd_set_option("total_deal_df", total_deal_df, 10)

In [19]:
# Перенос сделок из [total_deal_df] на MT5server методом [DealPerform], для балансовых сделок используется другой метод <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
attributes1 = ['crm_deals_id', 'open_deals_id', 'close_deals_id', 'positions_id', 'account_id', 'balance_transactions', 'comment', 'TimeMsc', 'Time', 'finance_type']
attributes2 = ['account_id', 'symbol', 'volume_lots', 'command', 'spread_x', 'open_price', 'close_price', 'open_time', 'close_time', 'rate_profit', 'mapping', 'profit']
attributes3 = ['type', 'CloseTime', 'CloseTimeMsc', 'symbol_digits', 'position_id', 'deals_swap']
attributes = attributes1 + attributes2 + attributes3
attributes = pd.Series(attributes)

#if manager.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password):
#pump_mode = manager.EnPumpModes.PUMP_MODE_POSITIONS
manager = manager_connect(pump_mode = manager.EnPumpModes.PUMP_MODE_POSITIONS) # = manager.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password)
if manager:
    print("Успешное соеденение MT5 manager")
    b, t, e = 0,0,0
    for index, row in total_deal_df.iterrows():                                                 # Проход по строкам DataFrame сверху вниз

        input("Нажмите Enter для продолжения...")
        local_vars = {}                                                                         # Создание словаря для хранения значений

        for attr in attributes:                                                                 # Присваивание значений переменным в словарь
            local_vars[attr] = row[attr]

        type_t = local_vars['type']                                                             # Определяем тип операции торговая / балансовая
        if type_t == "balance":                                                                 #print("\n Балансовая Операция ----------------

            user = MT5Manager.MTUser(manager) 
            login_user = local_vars['account_id'] 
            user.Login = login_user

            finance_type = local_vars['finance_type'] 
            balance_transactions = local_vars['balance_transactions']

            if finance_type == 'correction' or finance_type == 'credit':                        # Определяем тим балансовой операции                 
                if finance_type == 'correction':
                    deal_action = MT5Manager.MTDeal.EnDealAction.DEAL_CORRECTION
                else:
                    deal_action = MT5Manager.MTDeal.EnDealAction.DEAL_CREDIT
            elif finance_type == 'bonus' or finance_type == 'deposit' or finance_type == 'transfer_in':
                if finance_type == 'bonus':
                    deal_action = MT5Manager.MTDeal.EnDealAction.DEAL_BONUS
                else:
                    deal_action = MT5Manager.MTDeal.EnDealAction.DEAL_BALANCE
            elif (finance_type == 'withdrawal') or (finance_type == 'transfer_out'):
                deal_action = MT5Manager.MTDeal.EnDealAction.DEAL_BALANCE
                balance_transactions *= -1
            else:
                deal_action = None
                print(f"ERROR; row = {index}; Неизвестный тип операции, {finance_type}")                    # фиксируем неизвестный тип балансовой операции
                error_operatio_balance_generis_df.at[index, 'mt5error'] = f'ERROR; row = {index}; Неизвестный тип операции, {finance_type}'
                error_operatio_balance_generis_df  = pd.concat([error_operatio_balance_generis_df , pd.DataFrame([row])], ignore_index=True)

            if deal_action != None:
                if local_vars['comment'] == "referrals":
                    comment = str(f"#{local_vars['crm_deals_id']}, {type_t}, referrals")
                else:
                    comment = str(f"#{local_vars['crm_deals_id']}, {type_t}, {finance_type}")


                deal_id = process_deal(user.Login, balance_transactions, deal_action, comment)              # запускаем балансовую функцию
                
                
                
                if deal_id:
                                                         
                    deal = manager.DealRequest(deal_id)

                    total_deal_df.at[index, 'open_deals_id'] = deal.Deal                                    # фиксируем номер сделки на стороне MT5

                    a = np.int64(local_vars['Time'])
                    deal.Time = int(local_vars['Time'])
                    deal.TimeMsc = int(local_vars['TimeMsc'])
                    update_result = manager.DealUpdate(deal)

                    if update_result > 1:                                                                   # фиксируем невозможность обновить дату балансовой операции
                        #input("Нажмите Enter, чтобы продолжить...")
                        print(f"Error updating deal at Time {deal.Time}: {MT5Manager.LastError()}")
                        error_operatio_balance_generis_df.at[index, 'mt5error'] = f"Error updating deal at Time {deal.Time}: {MT5Manager.LastError()}"
                        error_operatio_balance_generis_df = pd.concat([error_update_df, pd.DataFrame([row])], ignore_index=True)
                    """
                    for attr in attributes:
                        print(f"{attr}={local_vars[attr]}", end=", ")
                    print()"""
                    
                    del deal

                else:
                    print(print(f"Error создание объекта БАЛАНСОВОЙ сделки {MT5Manager.LastError()}"))                # фиксируем невозможность создать объект балансовой операции
                    total_deal_df.at[index, 'mt5error'] = f"Error создание объекта БАЛАНСОВОЙ сделки {MT5Manager.LastError()}"
                    row_to_add = total_deal_df.loc[[index]]
                    error_deal_balance_df = pd.concat([error_deal_balance_df, row_to_add], ignore_index=True)

                #input("Нажмите Enter, чтобы продолжить...")
            del user

            b += 1
            print(f"balance: {b}, trade: {t}, error: {e}")

        elif type_t == "trade":
            #print("\n Торговая Операция ------------------------------------------------------------------------")
            deal = MT5Manager.MTDeal(manager)                                                           # Создаем объект сделки напрямую
                                                                                                        # общие атрибуты
            login         = local_vars['account_id']
            symbol        = local_vars['mapping']
            symbol_digits = local_vars['symbol_digits']

            if pd.isna(symbol):                                                                         # Проверка на наличие символа для открытия позиции
                print(f"ERROR: mapping = {symbol}; symbol = {local_vars['symbol']}; login = {login}")
                total_deal_df.at[index, 'mt5error'] = f"Error отсутствие СИМВОЛА {MT5Manager.LastError()}"
                row_to_add = total_deal_df.loc[[index]].dropna(axis=1, how='all')
                error_symbol_df = pd.concat([error_symbol_df, row_to_add], ignore_index=True)

            else:                                                                                       # Символ для открытия на МТ присутствует
                rate_profit = local_vars['rate_profit']                                               
                volume = int(local_vars['volume_lots'] * 10000)                                         # Переводим объём в метрику МТ5
                deal.Login          = login
                deal.Symbol         = symbol
                deal.Volume         = volume
                #print(symbol, "symbol_digits = ",symbol_digits)
                deal.Digits         = int(symbol_digits)
                                                                                                        # атрибуты открытия
                action_open         = local_vars['command']
                #print(symbol, "action_open = ",action_open)
                price_open          = local_vars["open_price"]
                time_open           = local_vars["Time"]
                timemsc_open        = local_vars["TimeMsc"]
                commission_spread   = local_vars['spread_x']
                deals_swap          = local_vars['deals_swap']
                position_id_crm     = local_vars["position_id"]
                #print(symbol, "deals_swap = ",deals_swap)
                print(f"{symbol}, volume = {volume}, Digits = {symbol_digits}, action = {action_open}, price = {price_open}, time = {time_open}, timemsc = {timemsc_open}, commission = {commission_spread}, rate_profit = {rate_profit}")
                
                deal.Action         = action_open
                deal.Price          = price_open
                deal.Time           = time_open
                deal.TimeMsc        = timemsc_open
                #deal.Commission     = commission_spread
                #deal.RateProfit     = rate_profit
                deal.Comment        = str(f"#{local_vars['crm_deals_id']}, RP={np.round(rate_profit, 2)}, SP={commission_spread}")

                iter = 0
                seconds_delay = 0.01
                pos_id = -1

                network_error_mt5 = "(-1, <EnMTAPIRetcode.MT_RET_ERR_NETWORK: 7>, 'Network error')"
                error_mt5 = network_error_mt5
                while error_mt5 == network_error_mt5:
                    if seconds_delay > 300:  
                        print(f"Превышено время ожидания [{seconds_delay}] секунд; Выход из цикла.")
                        break                           # Прерываем цикл после 60 секунд
                    if iter > 0:
                        print(f"ЗАдержка перед .DealPerform(deal) [{seconds_delay}] секунд")
                        time.sleep(seconds_delay)
                        seconds_delay *= 2
                    if not manager.DealPerform(deal):                                               # OPEN IN DIALS 
                        error_mt5 = MT5Manager.LastError()                                             
                        print("ERROR: OPEN;",  MT5Manager.LastError())
                    else:
                        pos_id = deal.PositionID                                                    # Фиксируем номер позиции на стороне MT5
                        deal_id = deal.Deal
                        total_deal_df.at[index, 'open_deals_id'] = deal_id                        # фиксируем номер СДЕЛКИ на стороне MT5
                        total_deal_df.at[index, 'positions_id']  = pos_id                           # фиксируем номер ПОЗИЦИИ на стороне MT5
                        error_mt5 = "zerro"
                    iter +=1

                if pos_id > 0:
                                                                                
                    pos_update  = False
                    #deal        = manager.DealRequest(deal_id)

                    if deal is not None:
                        pos_id = deal.PositionID                                                    # Фиксируем номер позиции на стороне MT5
                        #print(f"Строка [{index}] из [{len_df}]; Получен Идентификатор позиции счёта {login} открытый сделкой[{deal_id}] pos_id = {pos_id}")

                        on_break = "N"
                        on_break    = str(input(f"deals_swap = {deals_swap}, deal_i = {deal_id}, Ввдите значение on_break ="))
                        if (on_break == "Y") | ( on_break == "y"):
                            print(f"Введено значение on_break = {on_break} => STOP перебора ДФ и обновления SWOP")
                            break

                        if ((deals_swap > 0 or deals_swap < 0) and  position_id_crm == 1):
                            try:
                                seconds_delay = 0                                                   # Принудительная задержка для получения информации о позиции
                                pos = False
                                while pos == False:
                                    if seconds_delay > 6:                                       
                                        print(f"Превышено время ожидания [{seconds_delay}] секунд; Выход из цикла.")
                                        print(f"ERROR: SWOP позиции [{pos_id}] по сделке [{deal_id}] не обновлён")
                                        break                                                       # Прерываем цикл
                                    if  seconds_delay > 0:                                          # Выводим текущую ошибку, повлекшую неудачу
                                        print(f"ERROR: SWOP {deals_swap} не установлен в позицию {pos_id}, {MT5Manager.LastError()}")
                                        
                                    print(f"ЗАдержка перед PositionGetByTicket({pos_id}); [{seconds_delay}] секунд")
                                    time.sleep(seconds_delay)
                                    pos = manager.PositionGetByTicket(pos_id)                       # Создаём объект позиции
                                    seconds_delay += 0.01
                                    seconds_delay *= 2
                                                                        
                                if pos:
                                    pos_storage = pos.Storage

                                    if pos.Storage > 0:                                             # Контроль Отсутствия установленного SWOP
                                        print(f"SWOP позиции [{pos_id}] по сделке [{deal_id}] отличен от Нуля и равен [{pos_storage}]")
                                    else:
                                        pos.Storage = deals_swap                                    # Устанавливаем SWOP в объект позиции
                                        print(f"SWOP [{pos.Storage}] установлен в объект позиции [{pos_id}] по сделке [{deal_id}")
                                else:
                                    print(f"ERROR: manager_connect.PositionGetByTicket({pos_id})")

                                if not manager.PositionUpdate(pos):                                 # Обновляем позицию
                                    print(f"ERROR: manager_connect.PositionUpdate({pos_id})")
                                    raise Exception("ERROR: SWOP. Position update failed")
                                else:
                                    print(f"SWOP позиции [{pos_id}] по сделке [{deal_id}] ОБНОВЛЁН [{pos.Storage}]")
                                    pos_update = True

                                if pos_update == False:
                                    error_txt = f"{MT5Manager.LastError()}"
                                    deal_df.at[index, 'mt5error'] = error_txt
                                    row = deal_df.iloc[index]
                                    error_swop_df = pd.concat([error_swop_df, pd.DataFrame([row])], ignore_index=True)
                                
                                del pos

                            except Exception as ee:
                                    print(f"ERROR: SWOP {deals_swap} не установлен в позицию {pos_id}, {MT5Manager.LastError()}")
                    # >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

                    #print(f"deal.Login = {deal.Login}, deal.Symbol =  {deal.Symbol = }, volume = {volume}, deal.Digits = {deal.Digits}, deal.Action = {deal.Action},'deal.Price = {deal.Price }")
                    #print(f"deal.Time = {deal.Time}, deal.TimeMsc = {deal.TimeMsc}, deal.Commission = {deal.Commission}, deal.RateProfit = {deal.RateProfit}, deal.Comment = {deal.Comment}")
                    #input("Нажмите Enter, чтобы продолжить...")

                    #print(f"deal.PositionID = {pos_id}, open_deals_id = {deal.Deal}")

                    if position_id_crm == 3:                                  # определяем, на стороне CRM ПОЗИЦИЯ закрыта или нет

                        # атрибуты закрытия
                        action_close  = 1 if action_open == 0 else (0 if action_open == 1 else print("error"))
                        deal.Action   = action_close
                        price_close   = local_vars["close_price"]
                        time_close    = local_vars["CloseTime"]
                        timemsc_close = local_vars["CloseTimeMsc"]
                        deals_swap    = local_vars['deals_swap']

                        # создаём объект сделки для закрытия
                        #deal.RateProfit = rate_profit
                        deal.Price      = price_close
                        deal.Time       = time_close
                        deal.TimeMsc    = timemsc_close
                        deal.Commission = 0
                        deal.PositionID = pos_id
                        profit_crm      = round(local_vars["profit"] - local_vars["spread_x"], 2)
                        deal.Comment    = str(f"#{local_vars['crm_deals_id']}, PC={profit_crm}")

                        if deals_swap > 0 or deals_swap < 0: deal.Storage = deals_swap              # Установка SWOP

                        if manager.DealPerform(deal):
                            #input("Нажмите Enter, чтобы продолжить...")
                            pos_id = deal.PositionID
                            total_deal_df.at[index, 'close_deals_id'] = deal.Deal                   # фиксируем номер СДЕЛКИ на стороне MT5
                            
                            deal_profit = round(deal.Profit, 2)                                     # получаем зафиксированную прибыль по сделке
                            total_deal_df.at[index, 'deal_profit'] = deal_profit                    
                            deal_profit_dif = round(profit_crm - deal_profit, 2)
                            total_deal_df.at[index, 'deal_profit_dif'] = deal_profit_dif            # получаем разницу в прибыли на стороне МТ5 и в CRM

                            # Проверка прибыли позиции <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
                            #print(f'{print(pos_id)}, CLOSE: {deal.Print()},  profit_crm = {profit_crm}, deal_profit = {deal_profit}, deal_profit_dif = {deal_profit_dif}')
                            if abs(deal_profit) > 0:
                                profit_dif_per = profit_crm / deal_profit
                                total_deal_df.at[index, 'deal_profit_dif_per'] = profit_dif_per
                                
                                if profit_dif_per > 1.05 or profit_dif_per < 0.95:
                                    row = total_deal_df.iloc[index]
                                    error_deal_profit_df = pd.concat([error_deal_profit_df, pd.DataFrame([row])], ignore_index=True)
                            # >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
                        else:
                            print(f"CLOSE ERROR:{deal} {MT5Manager.LastError()}")
                            total_deal_df.at[index, 'mt5error'] = f"CLOSE ERROR:{deal} {MT5Manager.LastError()}"
                            row = total_deal_df.iloc[index]
                            error_deal_close_df = pd.concat([error_deal_close_df, pd.DataFrame([row])], ignore_index=True)
                else:
                    error_txt = f"OPEN ERROR: {login}, {deal.Print()} {MT5Manager.LastError()}"
                    print(error_txt)
                    total_deal_df.at[index, 'mt5error'] = error_txt
                    row = total_deal_df.iloc[index]
                    error_deal_open_df = pd.concat([error_deal_open_df, pd.DataFrame([row])], ignore_index=True)

                del deal, pos_id

                t += 1
                print(f"balance: {b}, trade: {t}, error: {e}")
                
        else:
            e += 1
            print(f"balance: {b}, trade: {t}, error: {e}")

print(f"manager_balance.Disconnect() = {manager.Disconnect()}") 

# Создание DF с итогами переноса позиций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
def error_txte(df_c):                                                           # Функция подсчёта количества ключевых сущностей таблицы
    if df_c.empty:
        return []
    elif df_c.apply(lambda x: isinstance(x, (int, np.integer))).all():          # Если все значения целые числа, возвращаем список чисел
        return {int(value) for value in set(df_c)}
    elif df_c.apply(lambda x: isinstance(x, str)).all():                        # Если все значения строки, возвращаем список строк
        return {str(value) for value in set(df_c)}
    else:
        return {str(value) for value in set(df_c)}                              # Если значения смешанные, возвращаем их как строки

error_set = error_txte(error_deal_profit_df["account_id"])
pd_set_option(f"\n Счетов с Сделками с отклонением от профита {len(error_set)} \n {error_set }",error_deal_profit_df, 3)
error_set = error_txte(error_symbol_df["symbol"])
pd_set_option(f"\n Счетов с Сделками по отсутствующим символам {len(error_set)} \n {error_set }",error_symbol_df, 3)
error_set = error_txte(error_operatio_balance_generis_df["account_id"])
pd_set_option(f"\n Счетов с неизвестными типами БАЛАНСОВЫХ операций {len(error_set)} \n {error_set }", error_operatio_balance_generis_df, 3)
error_set = error_txte(error_deal_balance_df["account_id"])
pd_set_option(f"\n Счета с НеСозданными БАЛАНСОВЫМИ объектами сделок {len(error_set)} \n {error_set }",error_deal_balance_df, 3)  
error_set = error_txte(error_update_balance_df["account_id"])
pd_set_option(f"\n Счета с Неудачными ОБНОВЛЕНИЯМИ транзакций {len(error_set)} \n {error_set }",error_update_balance_df, 3)

error_set = error_txte(error_swop_df["account_id"])
pd_set_option(f"\n Таблица НЕУДАЧНЫХ обновлений SWOP {len(error_set)} \n {error_set }",error_swop_df, 3)

error_set = error_txte(error_deal_open_df["account_id"])
pd_set_option(f"\n Счета с НеСозданными  объектами OPEN {len(error_set)} \n {error_set }",error_deal_open_df , 3)  
error_set = error_txte(error_deal_close_df["account_id"])
pd_set_option(f"\n Счета с НеСозданными  объектами CLOSE {len(error_set)} \n {error_set }",error_deal_close_df, 3)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Создание CSV с итогами переноса позиций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
error_symbol_df.to_csv(r'files\migration_errors\Таблица Отсутствующих символов.csv', index=False)
error_deal_profit_df.to_csv(r'files\migration_errors\Таблица Сделок с отклонением от профита.csv', index=False)
error_operatio_balance_generis_df.to_csv(r'files\migration_errors\Таблица НеИзвестных типов БАЛАНСОВЫХ операций.csv', index=False)
error_deal_balance_df.to_csv(r'files\migration_errors\Таблица НеСозданных БАЛАНСОВЫХ объектов сделки.csv', index=False)
error_update_balance_df.to_csv(r'files\migration_errors\Таблица Неудачных ОБНОВЛЕНИЙ транзакций.csv', index=False)
error_swop_df.to_csv(r'files\migration_errors\Таблица НЕУДАЧНЫХ обновлений SWOP.csv', index=False)
error_deal_open_df.to_csv(r'files\migration_errors\Таблица НеСозданных объектов OPEN сделки.csv', index=False)
error_deal_close_df.to_csv(r'files\migration_errors\Таблица Неудачных CLOSE транзакций.csv', index=False)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>


import.py manager =  <MT5Manager.ManagerAPI object at 0x000002E45BC260B0>
import.py MT5manager connect: True
Успешное соеденение MT5 manager
AUDUSD, volume = 10000, Digits = 5, action = 1, price = 0.6619, time = 1701706454, timemsc = 1701706454198, commission = 39.0, rate_profit = -0.00042302
balance: 0, trade: 1, error: 0
EOAN.XE, volume = 2000, Digits = 2, action = 0, price = 12.524, time = 1702387061, timemsc = 1702387061372, commission = -300.0, rate_profit = -106.42962963


C:\Users\nigilist\AppData\Local\Temp\ipykernel_10212\1622533547.py:274: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  error_deal_profit_df = pd.concat([error_deal_profit_df, pd.DataFrame([row])], ignore_index=True)


balance: 0, trade: 2, error: 0
LHAG.DE, volume = 300, Digits = 2, action = 0, price = 8.151, time = 1702656363, timemsc = 1702656363270, commission = -250.0, rate_profit = -27.69220152
Введено значение on_break = y => STOP перебора ДФ и обновления SWOP
manager_balance.Disconnect() = True

 Счетов с Сделками с отклонением от профита 1 
 {194863}


,symbol,position_id,command,re_open,comm,swap,point_profit,symbol_contract_size,volume_lots,rate_profit,calc_profit_2,calc_crm_profit_check_2,profit_crm,spread_x,profit,lead_id,account_id,leverage_x,margin,profit_right,point_profit_2,current_rate,open_price,close_price,open_time,close_time,mapping,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_calc_mode,Time,CloseTime,TimeMsc,CloseTimeMsc,type,deals_swap,balance_transactions,comment,finance_type,crm_deals_id,open_deals_id,close_deals_id,positions_id,mt5error,deal_profit,deal_profit_dif,deal_profit_dif_per
0,_E.ON,3,0,0,1,0.0,-0.135,100,0.2,-106.42963,287.36,9.995347e-10,287.36,-300.0,-12.64,194865,194863,1.0,270.23,-2.7,NaN,12.389,12.524,12.389,2023-12-12 15:17:41,2024-05-27 10:05:39,EOAN.XE,_E.ON,NaN,EUR,EUR,2,0,2.0,1702387061,1716793539,1702387061372,1716793539834,trade,-300.0,NaN,NaN,NaN,41253,4264406,4264407,4585801,"OPEN ERROR: 194863, #0 buy 0.2 EOAN.XE at 12.5...",-2.8,290.16,-102.628571



 Счетов с Сделками по отсутствующим символам 0 
 []


...

,symbol,position_id,command,re_open,comm,swap,point_profit,symbol_contract_size,volume_lots,rate_profit,calc_profit_2,calc_crm_profit_check_2,profit_crm,spread_x,profit,lead_id,account_id,leverage_x,margin,profit_right,point_profit_2,current_rate,open_price,close_price,open_time,close_time,mapping,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_calc_mode,Time,CloseTime,TimeMsc,CloseTimeMsc,type,deals_swap,balance_transactions,comment,finance_type,crm_deals_id,open_deals_id,close_deals_id,positions_id,mt5error,deal_profit,deal_profit_dif,deal_profit_dif_per



 Счета с НеСозданными  объектами OPEN 15 
 {215034, 225989, 207593, 235692, 226797, 194863, 234672, 75408, 241971, 246204, 110677, 235415, 97498, 262908, 234174}


,symbol,position_id,command,re_open,comm,swap,point_profit,symbol_contract_size,volume_lots,rate_profit,calc_profit_2,calc_crm_profit_check_2,profit_crm,spread_x,profit,lead_id,account_id,leverage_x,margin,profit_right,point_profit_2,current_rate,open_price,close_price,open_time,close_time,mapping,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_calc_mode,Time,CloseTime,TimeMsc,CloseTimeMsc,type,deals_swap,balance_transactions,comment,finance_type,crm_deals_id,open_deals_id,close_deals_id,positions_id,mt5error,deal_profit,deal_profit_dif,deal_profit_dif_per
0,AUD/USD,3,1,0,-1,0.0,0.6619,100000,1.00,-0.000423,-27.999694,3.062000e-04,-28.0,39.0,11.0,215036,215034,400.0,166.13,66190.0000,NaN,0.66218,0.6619,0.66218,2023-12-04 18:14:14,2023-12-04 18:20:07,AUDUSD,AUDUSD,Australian Dollar vs US Dollar,USD,AUD,5,0,0.0,1701706454,1701706807,1701706454198,1701706807457,trade,39.0,NaN,NaN,NaN,38481,-1,-1,-1,"OPEN ERROR: 215034, #0 sell 1 AUDUSD at 0.6619...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14,SZSE COMPONENT B,1,0,0,1,0.0,68.9986,100,0.01,1.000020,69.000000,-1.840611e-08,69.0,-6.0,63.0,235679,235692,100.0,78.56,68.9986,NaN,7924.59960,7855.6010,NaN,2025-01-14 16:29:26,NaN,399003,399003,NaN,USD,USD,2,0,2.0,1736864966,0,1736864966604,0,trade,-6.0,NaN,NaN,NaN,201795,-1,-1,-1,"OPEN ERROR: 235692, #0 buy 0.01 399003 at 7855...",NaN,NaN,NaN



 Счета с НеСозданными  объектами CLOSE 1 
 {215034}


,symbol,position_id,command,re_open,comm,swap,point_profit,symbol_contract_size,volume_lots,rate_profit,calc_profit_2,calc_crm_profit_check_2,profit_crm,spread_x,profit,lead_id,account_id,leverage_x,margin,profit_right,point_profit_2,current_rate,open_price,close_price,open_time,close_time,mapping,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_calc_mode,Time,CloseTime,TimeMsc,CloseTimeMsc,type,deals_swap,balance_transactions,comment,finance_type,crm_deals_id,open_deals_id,close_deals_id,positions_id,mt5error,deal_profit,deal_profit_dif,deal_profit_dif_per
0,AUD/USD,3,1,0,-1,0.0,0.6619,100000,1.0,-0.000423,-27.999694,0.000306,-28.0,39.0,11.0,215036,215034,400.0,166.13,66190.0,NaN,0.66218,0.6619,0.66218,2023-12-04 18:14:14,2023-12-04 18:20:07,AUDUSD,AUDUSD,Australian Dollar vs US Dollar,USD,AUD,5,0,0.0,1701706454,1701706807,1701706454198,1701706807457,trade,39.0,NaN,NaN,NaN,38481,4264402,-1,4585798,CLOSE ERROR:<MT5Manager.MTDeal object at 0x000...,NaN,NaN,NaN


In [ ]:
print(f"manager_balance.Disconnect() = {manager.Disconnect()}") 

In [ ]:
pd_set_option(f"\n Таблица НЕУДАЧНЫХ обновлений SWOP {len(error_set)} \n {error_set }",error_swop_df, 3)

In [ ]:
print(f"manager_balance.Disconnect() = {manager.Disconnect()}") 

In [ ]:
filtr_df = total_deal_df[total_deal_df["crm_deals_id"] == 115419]
pd_set_option("ЗАПРОС ПОКОНКРЕТНОМУ ОРДЕРУ", filtr_df, 3)




In [26]:
df_to_csv(total_deal_df, "total_deal_df_final_20250128.csv")

Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\total_deal_df_final_20250128.csv


In [ ]:

pd_set_option("", total_deal_df, 10)

In [33]:
# Функция подсчёта количества ключевых сущностей таблицы <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
def error_txte(df_c):                                                                                               
    if df_c.empty:                                                     return []
    elif df_c.apply(lambda x: isinstance(x, (int, np.integer))).all(): return {int(value) for value in set(df_c)}   # Если все значения целые числа, возвращаем список чисел
    elif df_c.apply(lambda x: isinstance(x, str)).all():               return {str(value) for value in set(df_c)}   # Если все значения строки, возвращаем список строк
    else:   return {str(value) for value in set(df_c)}   # Если значения смешанные, возвращаем их как строки

In [ ]:
for index, row in error_deal_open_df.iterrows():
    print(f"Индекс: {index}, Ошибка: {row['mt5error']}")

error_acc_set = error_txte(error_deal_open_df["account_id"])
error_symbol_set = error_txte(error_deal_open_df["mapping"])

print(f"\n Счетов с НЕ созданными сделками [{len(error_acc_set)}]: \n {error_acc_set };")
print(f"\n Инструменты по которым не удалось создать сделки [{len(error_symbol_set)}]: \n {error_symbol_set };")

error_set = error_txte(error_deal_open_df["account_id"])
pd_set_option(f"\n Таблица с НеСозданными  объектами OPEN",error_deal_open_df , 50)

In [ ]:
error_set = error_txte(error_deal_profit_df["account_id"])
pd_set_option(f"\n Счетов с Сделками с отклонением от профита {len(error_set)} \n {error_set }",error_deal_profit_df)

In [ ]:
# Ошибки по сделкам ЗАКРЫТИЯ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ```````````````````````````````````````````````````````````````````````````````````````````````````````````````
for index, row in error_deal_close_df.iterrows():
    print(f"Индекс: {index}, Ошибка: {row['mt5error']}")
error_set = error_txte(error_deal_close_df["account_id"])
pd_set_option(f"\n Таблица Неудачных CLOSE транзакций {len(error_set)} \n {error_set }",error_deal_close_df, 50)

In [ ]:
for index, row in total_deal_df.iterrows():
    print(f"Индекс: {index}, Ошибка: {row['mt5error']}")
error_set = error_txte(total_deal_df["account_id"])
sorted_df = total_deal_df.sort_values(by='positions_id', ascending=True)
pd_set_option(f"\n Таблица Неудачных CLOSE транзакций {len(error_set)} \n {error_set }", total_deal_df, rows = 50)

In [ ]:
total_deal_df.to_csv('total_deal_df.csv', index=False)  # index=False исключает запись индекса в файл

In [ ]:
# DF с итогами переноса позиций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
error_set = error_txte(error_deal_profit_df["account_id"])
pd_set_option(f"\n Счетов с Сделками с отклонением от профита {len(error_set)} \n {error_set }",error_deal_profit_df, 50)
error_set = error_txte(error_symbol_df["symbol"])
pd_set_option(f"\n Таблица Отсутствующих символов {len(error_set)} \n {error_set }",error_symbol_df, 3)
error_set = error_txte(error_operatio_balance_generis_df["account_id"])
pd_set_option(f"\n Таблица НеИзвестных типов БАЛАНСОВЫХ операций {len(error_set)} \n {error_set }", error_operatio_balance_generis_df, 50)
error_set = error_txte(error_deal_balance_df["account_id"])
pd_set_option(f"\n Таблица НеСозданных БАЛАНСОВЫХ объектов сделки {len(error_set)} \n {error_set }",error_deal_balance_df, 3)  
error_set = error_txte(error_update_balance_df["account_id"])
pd_set_option(f"\n Таблица Неудачных ОБНОВЛЕНИЙ транзакций {len(error_set)} \n {error_set }",error_update_balance_df, 3)
error_set = error_txte(error_deal_open_df["account_id"])
pd_set_option(f"\n Таблица НеСозданных объектов OPEN сделки {len(error_set)} \n {error_set }",error_deal_open_df , 50)  
error_set = error_txte(error_deal_close_df["account_id"])
pd_set_option(f"\n Таблица Неудачных CLOSE транзакций {len(error_set)} \n {error_set }",error_deal_close_df, 3)

In [13]:
total_deal_df.to_csv(r'files\migration_errors\total_deal_df_17.csv', index=False)

In [ ]:
# Создание CSV с итогами переноса позиций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

error_deal_profit_df.to_csv(r'files\migration_errors\Таблица Сделок с отклонением от профита.csv', index=False)
error_symbol_df.to_csv(r'files\migration_errors\Таблица Отсутствующих символов.csv', index=False)
error_operatio_balance_generis_df.to_csv(r'files\migration_errors\Таблица НеИзвестных типов БАЛАНСОВЫХ операций.csv', index=False)
error_deal_balance_df.to_csv(r'files\migration_errors\Таблица НеСозданных БАЛАНСОВЫХ объектов сделки.csv', index=False)
error_update_balance_df.to_csv(r'files\migration_errors\Таблица Неудачных ОБНОВЛЕНИЙ транзакций.csv', index=False)
error_deal_open_df.to_csv(r'files\migration_errors\Таблица НеСозданных объектов OPEN сделки.csv', index=False)
error_deal_close_df.to_csv(r'files\migration_errors\Таблица Неудачных CLOSE транзакций.csv', index=False)

In [ ]:
print(f"manager_balance.Disconnect() = {manager.Disconnect()}") 
print(f"admin_balance.Disconnect() = {admin.Disconnect()}") 

In [ ]:


error_set = error_txte(error_deal_profit_df["account_id"])
pd_set_option(f"\n Счетов с Сделками с отклонением от профита {len(error_set)} \n {error_set }",error_deal_profit_df, 50)